# JJP Image Studio Commercial v7

Use **Image_Studio_Commercial_v7.ipynb** for new music-video artwork. This replaces the v1–v6 image profile. Those historical notebooks include InsightFace research-only weights and must not be used as the commercial face workflow.

1. Import **Image_Studio_Commercial_Model_Packs.ipynb** into Kaggle. Use CPU + Internet. Run once with `PACK='create_edit'`, save outputs, then once with `PACK='finish'`. Existing Kaggle credentials are used to submit each as a private Dataset. Nothing is downloaded to your PC. See MODEL_PACKS.md.
2. Import the v7 studio notebook into Kaggle. Attach both private datasets (or the saved pack notebook outputs), select a GPU, enable Internet and run its single code cell. It verifies every model before installation. Missing packs produce a clear error without starting model downloads. Set `ENABLE_CREATE_EDIT=False` for an upscaling/compositing-only session.
3. Open the printed Cloudflare URL. In the workflow menu choose **Image Studio - …**, upload references and press Run. Startup does not generate automatically. Keep the startup cell running. The URL gives access to this session; keep it private.
4. Save Version with outputs before ending your session. Preserve approved cast, original photos, clean pre-swap targets, PNG metadata, provenance.json and licenses. GitHub updates do not update an already imported Kaggle notebook.

Recommended dedicated head replacement: **BFS Head Swap 4B**, with the author’s MIT-declared BFS adapter and Apache-licensed Klein 4B base. It replaces head/hair as well as face; inspect likeness and framing. Generic Identity Edit remains available for prompt-directed face changes. No head-swap quality or T4 speed ranking has been measured here.

Included: FLUX.2 Klein **4B distilled** generation, native single/multiple-reference editing, masked editing, two reference identity editing, CPU mesh face compositing, Real-ESRGAN 2×/4×, SeedVR2 restoration, approved cast, eight JJP episode worlds, custom world/palette/wing controls, deterministic branded square/landscape/vertical exports. All generation uses visible connected native ComfyUI nodes. No FaceID, ArcFace, InsightFace, GFPGAN or CodeFormer weights are silently downloaded.

Identity editing is generative: inspect facial likeness, hands, clothing and background changes. Describe image 1 as the target and image 2 as the identity reference; identify which person to change. For duets, map images 2 and 3 to Lead One and Lead Two explicitly. It is not a measured substitute for a neural face-swap identity metric. The mesh compositor is supplemental and showed mouth/edge artifacts in the CPU sample; it is not the recommended quality replacement. It transfers source pixels with landmarks; choose similar pose, expression and lighting. Indices are zero-based, left to right. Large pose changes, occlusion, hair and motion remain limitations. This notebook processes stills, not a temporally consistent video face-swap.

The distilled model uses four steps and CFG 1. Negative text in the JJP brief is retained for review but is not active negative guidance at CFG 1. Masked edit first generates a reference edit, then composites it within your painted LoadImage mask. Use MaskEditor and a white replacement mask. This preserves the unmasked target except for the selected feather band; it does not use a dedicated inpainting checkpoint. Export resizing is interpolation, not neural upscaling. Use the upscaling workflow for learned detail first.

Default weights need about 20.53 GB across two packs, plus runtime disk. Generation uses GPU 0; two T4s do not form a single shared VRAM pool. Low-VRAM/FP16 flags are supplied. Full FLUX inference on Kaggle/T4, peak memory, image quality and live Cloudflare remain unverified here. Do not treat model-author speed/memory claims as measured notebook results. See VALIDATION.md and LICENSE_AUDIT.md.


In [ ]:
# Single-cell startup. No image generation starts automatically.
from pathlib import Path
import sys, os, subprocess, json, time, urllib.request, hashlib, shutil, re, signal, fcntl
ENABLE_CREATE_EDIT = True
ENABLE_SEEDVR2 = True
ENABLE_MESH_SWAP = True
ENABLE_BFS_HEADSWAP = True
REQUIRE_ATTACHED_MODELS = True
PACK_BUILDER_SOURCE = '# CPU-only model pack builder. Run once and save the outputs on Kaggle.\nPACK_BUILDER_VERSION = \'dataset-upload-v3-direct-create\'\nAUTO_CREATE_DATASET = True  # Uses your existing Kaggle authentication to create a private Dataset.\nDATASET_OWNER = \'\'  # Usually detected from authenticated Kaggle API credentials.\nPACK = \'create_edit\'  # Run once per pack: create_edit (~16.13 GB), finish (~4.40 GB)\nfrom pathlib import Path\nimport os, json, hashlib, shutil, time, urllib.request, urllib.error, fcntl\nCATALOG = {\'profile\': \'commercial-v7\', \'reviewed\': \'2026-11-09\', \'assets\': [{\'pack\': \'create_edit\', \'folder\': \'diffusion_models\', \'name\': \'flux-2-klein-4b.safetensors\', \'bytes\': 7751105712, \'sha256\': \'ec3d4e733a771f61c052fb4856c48b336c55eaf2c65487c2a1faeb9bbda7a343\', \'license\': \'Apache-2.0\', \'source\': \'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/diffusion_models/flux-2-klein-4b.safetensors\'}, {\'pack\': \'create_edit\', \'folder\': \'text_encoders\', \'name\': \'qwen_3_4b.safetensors\', \'bytes\': 8044982048, \'sha256\': \'6c671498573ac2f7a5501502ccce8d2b08ea6ca2f661c458e708f36b36edfc5a\', \'license\': \'Apache-2.0\', \'source\': \'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/text_encoders/qwen_3_4b.safetensors\'}, {\'pack\': \'create_edit\', \'folder\': \'vae\', \'name\': \'flux2-vae.safetensors\', \'bytes\': 336211292, \'sha256\': \'868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3\', \'license\': \'Apache-2.0\', \'source\': \'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/vae/flux2-vae.safetensors\'}, {\'pack\': \'finish\', \'folder\': \'upscale_models\', \'name\': \'RealESRGAN_x4plus.pth\', \'bytes\': 67040989, \'sha256\': \'4fa0d38905f75ac06eb49a7951b426670021be3018265fd191d2125df9d682f1\', \'license\': \'BSD-3-Clause\', \'source\': \'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth\'}, {\'pack\': \'finish\', \'folder\': \'diffusion_models\', \'name\': \'seedvr2_3b_int8_convrot.safetensors\', \'bytes\': 3458259704, \'sha256\': \'c3dec8bcc5916843a8a858572970597462e1f2dc598d6dfd818f6cd40f53a157\', \'license\': \'Apache-2.0\', \'source\': \'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/diffusion_models/seedvr2_3b_int8_convrot.safetensors\'}, {\'pack\': \'finish\', \'folder\': \'vae\', \'name\': \'seedvr2_ema_vae_fp16.safetensors\', \'bytes\': 501324814, \'sha256\': \'20678548f420d98d26f11442d3528f8b8c94e57ee046ef93dbb7633da8612ca1\', \'license\': \'Apache-2.0\', \'source\': \'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/vae/seedvr2_ema_vae_fp16.safetensors\'}, {\'pack\': \'finish\', \'folder\': \'mediapipe\', \'name\': \'face_landmarker.task\', \'bytes\': 3758596, \'sha256\': \'64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff\', \'license\': \'Apache-2.0\', \'source\': \'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task\'}, {\'pack\': \'finish\', \'folder\': \'loras\', \'name\': \'bfs_head_v1_flux-klein_4b.safetensors\', \'bytes\': 369644384, \'sha256\': \'77aabcd6144612dd4873080b99283dd9a27977712548bee161894f62c91ff09c\', \'license\': \'MIT\', \'source\': \'https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/bfs_head_v1_flux-klein_4b.safetensors\'}]}\nLICENSE_FILES = {\'realesrgan-BSD.txt\': \'BSD 3-Clause License\\n\\nCopyright (c) 2021, Xintao Wang\\nAll rights reserved.\\n\\nRedistribution and use in source and binary forms, with or without\\nmodification, are permitted provided that the following conditions are met:\\n\\n1. Redistributions of source code must retain the above copyright notice, this\\n   list of conditions and the following disclaimer.\\n\\n2. Redistributions in binary form must reproduce the above copyright notice,\\n   this list of conditions and the following disclaimer in the documentation\\n   and/or other materials provided with the distribution.\\n\\n3. Neither the name of the copyright holder nor the names of its\\n   contributors may be used to endorse or promote products derived from\\n   this software without specific prior written permission.\\n\\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\\n\', \'mediapipe-model-evidence.txt\': \'MediaPipe model license evidence, reviewed 2026-11-09. Bundle SHA256: 64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff\\n\\nhttps://storage.googleapis.com/mediapipe-assets/Model%20Card%20MediaPipe%20Face%20Mesh%20V2.pdf\\nGeng Yan, Google\\nIvan Grishchenko, Google\\nMODEL DATE\\nSeptember 15, 2022\\nLICENSED UNDER\\nApache License, Version 2.0\\n\\n_\\nBased on Model Cards for Model Repo ing, In Proceedings of FAT* Conference (FAT*2019). ACM, New York, NY, USA\\n\\nhttps://storage.googleapis.com/mediapipe-assets/MediaPipe%20BlazeFace%20Model%20Card%20(Short%20Range).pdf\\n\\nDOCUMENTATION\\nPaper: https://arxiv.org/abs/2006.10204\\n\\nLICENSED UNDER\\nApache License, Version 2.0\\n\\nBased on Model Cards for Model Reporting, In Proceedings of FAT* Conference (FAT*2019). ACM, New York, NY, USA\\n\\n\\nhttps://storage.googleapis.com/mediapipe-assets/Model%20Card%20Blendshape%20V2.pdf\\nAppendix.\\n\\nfront-facing camera\\n\\nLICENSED UNDER\\nApache License, Version 2.0\\nLe : Input frame with predicted facial landmarks.\\nRight: 3D avatar controlled with predicted blendshape\\ncoe cients\\n\', \'flux2-klein4b-APACHE.txt\': \'                             Apache License\\n                       Version 2.0, January 2004\\n                    http://www.apache.org/licenses/\\n\\nTERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\\n\\n1. Definitions.\\n\\n  "License" shall mean the terms and conditions for use, reproduction,\\n  and distribution as defined by Sections 1 through 9 of this document.\\n\\n  "Licensor" shall mean the copyright owner or entity authorized by\\n  the copyright owner that is granting the License.\\n\\n  "Legal Entity" shall mean the union of the acting entity and all\\n  other entities that control, are controlled by, or are under common\\n  control with that entity. For the purposes of this definition,\\n  "control" means (i) the power, direct or indirect, to cause the\\n  direction or management of such entity, whether by contract or\\n  otherwise, or (ii) ownership of fifty percent (50%) or more of the\\n  outstanding shares, or (iii) beneficial ownership of such entity.\\n\\n  "You" (or "Your") shall mean an individual or Legal Entity\\n  exercising permissions granted by this License.\\n\\n  "Source" form shall mean the preferred form for making modifications,\\n  including but not limited to software source code, documentation\\n  source, and configuration files.\\n\\n  "Object" form shall mean any form resulting from mechanical\\n  transformation or translation of a Source form, including but\\n  not limited to compiled object code, generated documentation,\\n  and conversions to other media types.\\n\\n  "Work" shall mean the work of authorship, whether in Source or\\n  Object form, made available under the License, as indicated by a\\n  copyright notice that is included in or attached to the work\\n  (an example is provided in the Appendix below).\\n\\n  "Derivative Works" shall mean any work, whether in Source or Object\\n  form, that is based on (or derived from) the Work and for which the\\n  editorial revisions, annotations, elaborations, or other modifications\\n  represent, as a whole, an original work of authorship. For the purposes\\n  of this License, Derivative Works shall not include works that remain\\n  separable from, or merely link (or bind by name) to the interfaces of,\\n  the Work and Derivative Works thereof.\\n\\n  "Contribution" shall mean any work of authorship, including\\n  the original version of the Work and any modifications or additions\\n  to that Work or Derivative Works thereof, that is intentionally\\n  submitted to Licensor for inclusion in the Work by the copyright owner\\n  or by an individual or Legal Entity authorized to submit on behalf of\\n  the copyright owner. For the purposes of this definition, "submitted"\\n  means any form of electronic, verbal, or written communication sent\\n  to the Licensor or its representatives, including but not limited to\\n  communication on electronic mailing lists, source code control systems,\\n  and issue tracking systems that are managed by, or on behalf of, the\\n  Licensor for the purpose of discussing and improving the Work, but\\n  excluding communication that is conspicuously marked or otherwise\\n  designated in writing by the copyright owner as "Not a Contribution."\\n\\n  "Contributor" shall mean Licensor and any individual or Legal Entity\\n  on behalf of whom a Contribution has been received by Licensor and\\n  subsequently incorporated within the Work.\\n\\n2. Grant of Copyright License. Subject to the terms and conditions of\\n  this License, each Contributor hereby grants to You a perpetual,\\n  worldwide, non-exclusive, no-charge, royalty-free, irrevocable\\n  copyright license to reproduce, prepare Derivative Works of,\\n  publicly display, publicly perform, sublicense, and distribute the\\n  Work and such Derivative Works in Source or Object form.\\n\\n3. Grant of Patent License. Subject to the terms and conditions of\\n  this License, each Contributor hereby grants to You a perpetual,\\n  worldwide, non-exclusive, no-charge, royalty-free, irrevocable\\n  (except as stated in this section) patent license to make, have made,\\n  use, offer to sell, sell, import, and otherwise transfer the Work,\\n  where such license applies only to those patent claims licensable\\n  by such Contributor that are necessarily infringed by their\\n  Contribution(s) alone or by combination of their Contribution(s)\\n  with the Work to which such Contribution(s) was submitted. If You\\n  institute patent litigation against any entity (including a\\n  cross-claim or counterclaim in a lawsuit) alleging that the Work\\n  or a Contribution incorporated within the Work constitutes direct\\n  or contributory patent infringement, then any patent licenses\\n  granted to You under this License for that Work shall terminate\\n  as of the date such litigation is filed.\\n\\n4. Redistribution. You may reproduce and distribute copies of the\\n  Work or Derivative Works thereof in any medium, with or without\\n  modifications, and in Source or Object form, provided that You\\n  meet the following conditions:\\n\\n  (a) You must give any other recipients of the Work or\\n      Derivative Works a copy of this License; and\\n\\n  (b) You must cause any modified files to carry prominent notices\\n      stating that You changed the files; and\\n\\n  (c) You must retain, in the Source form of any Derivative Works\\n      that You distribute, all copyright, patent, trademark, and\\n      attribution notices from the Source form of the Work,\\n      excluding those notices that do not pertain to any part of\\n      the Derivative Works; and\\n\\n  (d) If the Work includes a "NOTICE" text file as part of its\\n      distribution, then any Derivative Works that You distribute must\\n      include a readable copy of the attribution notices contained\\n      within such NOTICE file, excluding those notices that do not\\n      pertain to any part of the Derivative Works, in at least one\\n      of the following places: within a NOTICE text file distributed\\n      as part of the Derivative Works; within the Source form or\\n      documentation, if provided along with the Derivative Works; or,\\n      within a display generated by the Derivative Works, if and\\n      wherever such third-party notices normally appear. The contents\\n      of the NOTICE file are for informational purposes only and\\n      do not modify the License. You may add Your own attribution\\n      notices within Derivative Works that You distribute, alongside\\n      or as an addendum to the NOTICE text from the Work, provided\\n      that such additional attribution notices cannot be construed\\n      as modifying the License.\\n\\n  You may add Your own copyright statement to Your modifications and\\n  may provide additional or different license terms and conditions\\n  for use, reproduction, or distribution of Your modifications, or\\n  for any such Derivative Works as a whole, provided Your use,\\n  reproduction, and distribution of the Work otherwise complies with\\n  the conditions stated in this License.\\n\\n5. Submission of Contributions. Unless You explicitly state otherwise,\\n  any Contribution intentionally submitted for inclusion in the Work\\n  by You to the Licensor shall be under the terms and conditions of\\n  this License, without any additional terms or conditions.\\n  Notwithstanding the above, nothing herein shall supersede or modify\\n  the terms of any separate license agreement you may have executed\\n  with Licensor regarding such Contributions.\\n\\n6. Trademarks. This License does not grant permission to use the trade\\n  names, trademarks, service marks, or product names of the Licensor,\\n  except as required for reasonable and customary use in describing the\\n  origin of the Work and reproducing the content of the NOTICE file.\\n\\n7. Disclaimer of Warranty. Unless required by applicable law or\\n  agreed to in writing, Licensor provides the Work (and each\\n  Contributor provides its Contributions) on an "AS IS" BASIS,\\n  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\\n  implied, including, without limitation, any warranties or conditions\\n  of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\\n  PARTICULAR PURPOSE. You are solely responsible for determining the\\n  appropriateness of using or redistributing the Work and assume any\\n  risks associated with Your exercise of permissions under this License.\\n\\n8. Limitation of Liability. In no event and under no legal theory,\\n  whether in tort (including negligence), contract, or otherwise,\\n  unless required by applicable law (such as deliberate and grossly\\n  negligent acts) or agreed to in writing, shall any Contributor be\\n  liable to You for damages, including any direct, indirect, special,\\n  incidental, or consequential damages of any character arising as a\\n  result of this License or out of the use or inability to use the\\n  Work (including but not limited to damages for loss of goodwill,\\n  work stoppage, computer failure or malfunction, or any and all\\n  other commercial damages or losses), even if such Contributor\\n  has been advised of the possibility of such damages.\\n\\n9. Accepting Warranty or Additional Liability. While redistributing\\n  the Work or Derivative Works thereof, You may choose to offer,\\n  and charge a fee for, acceptance of support, warranty, indemnity,\\n  or other liability obligations and/or rights consistent with this\\n  License. However, in accepting such obligations, You may act only\\n  on Your own behalf and on Your sole responsibility, not on behalf\\n  of any other Contributor, and only if You agree to indemnify,\\n  defend, and hold each Contributor harmless for any liability\\n  incurred by, or claims asserted against, such Contributor by reason\\n  of your accepting any such warranty or additional liability.\\n\\nEND OF TERMS AND CONDITIONS\\n\', \'bfs-model-card-MIT.md\': \'Original model card snapshot, declared MIT. Source: https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/blob/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/README.md\\n\\n---\\nlicense: mit\\nbase_model:\\n- Qwen/Qwen-Image-2.1\\n- Qwen/Qwen-Image-Edit-2511\\n- black-forest-labs/FLUX.2-klein-9B\\n- black-forest-labs/FLUX.2-klein-4B\\n- black-forest-labs/FLUX.2-klein-base-4B\\n- black-forest-labs/FLUX.2-klein-base-9B\\n- black-forest-labs/FLUX.2-klein-9b-fp8\\n- black-forest-labs/FLUX.2-klein-4b-fp8\\n- krea/Krea-2-Raw\\npipeline_tag: image-to-image\\ntags:\\n- lora\\n- qwen-image\\n- qwen-image-2.1\\n- qwen-image-edit\\n- face-swap\\n- head-swap\\n- body-swap\\n- selective-body-swap\\n- image-editing\\n- comfyui\\n- bfs\\nlanguage:\\n- en\\nlibrary_name: diffusers\\n---\\n\\n# BFS: Best Face Swap\\n\\nBFS is a collection of LoRAs for face, head, and body replacement across Qwen Image 2.1, Qwen\\nImage Edit 2509/2511, Flux 2 Klein, Krea 2, and experimental LTX-2 workflows. Each release has\\nits own base-model, input-order, prompt, and workflow contract. Use the guide for the exact\\nversion you are loading.\\n\\n## Guides\\n\\n- [BFS Head V1 / V1.1 for Qwen Image 2.1](docs/qwen-image-2.1.md)\\n- [BFS Body Swap V1.0 for Qwen Image 2.1](docs/qwen-image-2.1-body.md)\\n- [BFS Body Swap Selective (Magenta) V1.0 for Qwen Image 2.1](docs/qwen-image-2.1-body-selective.md):\\n  replaces only the person marked in magenta, for group photos and busy scenes\\n- [Qwen Image Edit 2509/2511](docs/qwen-image-edit.md)\\n- [Flux 2 Klein](docs/flux-2-klein.md)\\n- [Krea 2](docs/krea-2.md)\\n- [Experimental LTX-2 weights](docs/ltx-2.md)\\n\\n## Downloads at a glance\\n\\n| Family | Weights | Workflow |\\n|---|---|---|\\n| Qwen Image 2.1 (head) | [`bfs_head_v1.1_qwen_2.1.safetensors`](bfs_head_v1.1_qwen_2.1.safetensors) (recommended), [`bfs_head_v1.1_alternative_qwen_2.1.safetensors`](bfs_head_v1.1_alternative_qwen_2.1.safetensors), [`bfs_head_v1_qwen_2.1.safetensors`](bfs_head_v1_qwen_2.1.safetensors) | [`Head Swap V1 Qwen 2.1 Workflow.json`](workflows/Head%20Swap%20V1%20Qwen%202.1%20Workflow.json) |\\n| Qwen Image 2.1 (body) | [`bfs_body_swap_v1.0_qwen_2.1.safetensors`](bfs_body_swap_v1.0_qwen_2.1.safetensors) | [`Body Swap Qwen Image 2.1 - V1.json`](workflows/Body%20Swap%20Qwen%20Image%202.1%20-%20V1.json), see the [body swap guide](docs/qwen-image-2.1-body.md) |\\n| Qwen Image 2.1 (selective body, magenta mark) | [`bfs_body_swap_selective_magenta_v1.0_qwen_2.1.safetensors`](bfs_body_swap_selective_magenta_v1.0_qwen_2.1.safetensors) | [`Body Swap Selective Qwen Image 2.1 - V1.0.json`](workflows/Body%20Swap%20Selective%20Qwen%20Image%202.1%20-%20V1.0.json), see the [selective guide](docs/qwen-image-2.1-body-selective.md) |\\n| Qwen Image Edit 2509 | [`bfs_face_v1_qwen_image_edit_2509.safetensors`](bfs_face_v1_qwen_image_edit_2509.safetensors), [`bfs_head_v1_qwen_image_edit_2509.safetensors`](bfs_head_v1_qwen_image_edit_2509.safetensors), [`bfs_head_v2_qwen_image_edit_2509.safetensors`](bfs_head_v2_qwen_image_edit_2509.safetensors), [`bfs_head_v3_qwen_image_edit_2509.safetensors`](bfs_head_v3_qwen_image_edit_2509.safetensors), [`bfs_head_v4_qwen_image_edit_2509.safetensors`](bfs_head_v4_qwen_image_edit_2509.safetensors) | See the [Qwen Edit guide](docs/qwen-image-edit.md) |\\n| Qwen Image Edit 2511 | [`bfs_head_v5_2511_original.safetensors`](bfs_head_v5_2511_original.safetensors), [`bfs_head_v5_2511_merged_version_rank_16_fp16.safetensors`](bfs_head_v5_2511_merged_version_rank_16_fp16.safetensors), [`bfs_head_v5_2511_merged_version_rank_32_fp32.safetensors`](bfs_head_v5_2511_merged_version_rank_32_fp32.safetensors) | See the [Qwen Edit guide](docs/qwen-image-edit.md) |\\n| Flux 2 Klein | [`bfs_head_v1_flux-klein_4b.safetensors`](bfs_head_v1_flux-klein_4b.safetensors), [`bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors`](bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors), [`bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors`](bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors), [`bfs_head_v1.1_optional_flux-klein_4b.safetensors`](bfs_head_v1.1_optional_flux-klein_4b.safetensors) | See the [Flux guide](docs/flux-2-klein.md) |\\n| Krea 2 | [`bfs_head_swap_v1.1_krea2.safetensors`](bfs_head_swap_v1.1_krea2.safetensors), [`bfs_body_swap_v1_krea2.safetensors`](bfs_body_swap_v1_krea2.safetensors) | See the [Krea 2 guide](docs/krea-2.md) |\\n| Experimental LTX-2 | [`bfs_head_swap_v1_ltx2_ic_lora_5000.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_5000.safetensors), [`bfs_head_swap_v1_ltx2_ic_lora_6000.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_6000.safetensors), [`bfs_head_swap_v1_ltx2_ic_lora_12000_train_3.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_12000_train_3.safetensors) | See the [LTX-2 guide](docs/ltx-2.md) |\\n\\n## General rules\\n\\n- Use the LoRA with its matching base model whenever possible.\\n- Follow the input order in the version guide; several BFS head releases intentionally use body\\n  first and reference face second.\\n- Start at LoRA strength `1.0`, then adjust for the base model, resolution, and reference image.\\n- The examples and workflows in this repository are demonstrations, not a guarantee of identical\\n  results on every image.\\n- Do not use or share results involving public figures or people who have not given consent.\\n\\n## Examples\\n\\nThe existing examples are grouped by workflow family in the individual guides. Qwen Image 2.1\\nexamples are `images/bfs_qwen_2.1_1.png` through `images/bfs_qwen_2.1_8.png`.\\n\\n## Support\\n\\nFeel free to experiment and share your results responsibly. For workflow issues, first update\\nComfyUI and verify that the base model, text encoder, VAE, workflow, and LoRA all belong to the\\nsame model family.\\n\', \'seedvr2-APACHE.txt\': \'                                 Apache License\\n                           Version 2.0, January 2004\\n                        http://www.apache.org/licenses/\\n\\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\\n\\n   1. Definitions.\\n\\n      "License" shall mean the terms and conditions for use, reproduction,\\n      and distribution as defined by Sections 1 through 9 of this document.\\n\\n      "Licensor" shall mean the copyright owner or entity authorized by\\n      the copyright owner that is granting the License.\\n\\n      "Legal Entity" shall mean the union of the acting entity and all\\n      other entities that control, are controlled by, or are under common\\n      control with that entity. For the purposes of this definition,\\n      "control" means (i) the power, direct or indirect, to cause the\\n      direction or management of such entity, whether by contract or\\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\\n      outstanding shares, or (iii) beneficial ownership of such entity.\\n\\n      "You" (or "Your") shall mean an individual or Legal Entity\\n      exercising permissions granted by this License.\\n\\n      "Source" form shall mean the preferred form for making modifications,\\n      including but not limited to software source code, documentation\\n      source, and configuration files.\\n\\n      "Object" form shall mean any form resulting from mechanical\\n      transformation or translation of a Source form, including but\\n      not limited to compiled object code, generated documentation,\\n      and conversions to other media types.\\n\\n      "Work" shall mean the work of authorship, whether in Source or\\n      Object form, made available under the License, as indicated by a\\n      copyright notice that is included in or attached to the work\\n      (an example is provided in the Appendix below).\\n\\n      "Derivative Works" shall mean any work, whether in Source or Object\\n      form, that is based on (or derived from) the Work and for which the\\n      editorial revisions, annotations, elaborations, or other modifications\\n      represent, as a whole, an original work of authorship. For the purposes\\n      of this License, Derivative Works shall not include works that remain\\n      separable from, or merely link (or bind by name) to the interfaces of,\\n      the Work and Derivative Works thereof.\\n\\n      "Contribution" shall mean any work of authorship, including\\n      the original version of the Work and any modifications or additions\\n      to that Work or Derivative Works thereof, that is intentionally\\n      submitted to Licensor for inclusion in the Work by the copyright owner\\n      or by an individual or Legal Entity authorized to submit on behalf of\\n      the copyright owner. For the purposes of this definition, "submitted"\\n      means any form of electronic, verbal, or written communication sent\\n      to the Licensor or its representatives, including but not limited to\\n      communication on electronic mailing lists, source code control systems,\\n      and issue tracking systems that are managed by, or on behalf of, the\\n      Licensor for the purpose of discussing and improving the Work, but\\n      excluding communication that is conspicuously marked or otherwise\\n      designated in writing by the copyright owner as "Not a Contribution."\\n\\n      "Contributor" shall mean Licensor and any individual or Legal Entity\\n      on behalf of whom a Contribution has been received by Licensor and\\n      subsequently incorporated within the Work.\\n\\n   2. Grant of Copyright License. Subject to the terms and conditions of\\n      this License, each Contributor hereby grants to You a perpetual,\\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\\n      copyright license to reproduce, prepare Derivative Works of,\\n      publicly display, publicly perform, sublicense, and distribute the\\n      Work and such Derivative Works in Source or Object form.\\n\\n   3. Grant of Patent License. Subject to the terms and conditions of\\n      this License, each Contributor hereby grants to You a perpetual,\\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\\n      (except as stated in this section) patent license to make, have made,\\n      use, offer to sell, sell, import, and otherwise transfer the Work,\\n      where such license applies only to those patent claims licensable\\n      by such Contributor that are necessarily infringed by their\\n      Contribution(s) alone or by combination of their Contribution(s)\\n      with the Work to which such Contribution(s) was submitted. If You\\n      institute patent litigation against any entity (including a\\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\\n      or a Contribution incorporated within the Work constitutes direct\\n      or contributory patent infringement, then any patent licenses\\n      granted to You under this License for that Work shall terminate\\n      as of the date such litigation is filed.\\n\\n   4. Redistribution. You may reproduce and distribute copies of the\\n      Work or Derivative Works thereof in any medium, with or without\\n      modifications, and in Source or Object form, provided that You\\n      meet the following conditions:\\n\\n      (a) You must give any other recipients of the Work or\\n          Derivative Works a copy of this License; and\\n\\n      (b) You must cause any modified files to carry prominent notices\\n          stating that You changed the files; and\\n\\n      (c) You must retain, in the Source form of any Derivative Works\\n          that You distribute, all copyright, patent, trademark, and\\n          attribution notices from the Source form of the Work,\\n          excluding those notices that do not pertain to any part of\\n          the Derivative Works; and\\n\\n      (d) If the Work includes a "NOTICE" text file as part of its\\n          distribution, then any Derivative Works that You distribute must\\n          include a readable copy of the attribution notices contained\\n          within such NOTICE file, excluding those notices that do not\\n          pertain to any part of the Derivative Works, in at least one\\n          of the following places: within a NOTICE text file distributed\\n          as part of the Derivative Works; within the Source form or\\n          documentation, if provided along with the Derivative Works; or,\\n          within a display generated by the Derivative Works, if and\\n          wherever such third-party notices normally appear. The contents\\n          of the NOTICE file are for informational purposes only and\\n          do not modify the License. You may add Your own attribution\\n          notices within Derivative Works that You distribute, alongside\\n          or as an addendum to the NOTICE text from the Work, provided\\n          that such additional attribution notices cannot be construed\\n          as modifying the License.\\n\\n      You may add Your own copyright statement to Your modifications and\\n      may provide additional or different license terms and conditions\\n      for use, reproduction, or distribution of Your modifications, or\\n      for any such Derivative Works as a whole, provided Your use,\\n      reproduction, and distribution of the Work otherwise complies with\\n      the conditions stated in this License.\\n\\n   5. Submission of Contributions. Unless You explicitly state otherwise,\\n      any Contribution intentionally submitted for inclusion in the Work\\n      by You to the Licensor shall be under the terms and conditions of\\n      this License, without any additional terms or conditions.\\n      Notwithstanding the above, nothing herein shall supersede or modify\\n      the terms of any separate license agreement you may have executed\\n      with Licensor regarding such Contributions.\\n\\n   6. Trademarks. This License does not grant permission to use the trade\\n      names, trademarks, service marks, or product names of the Licensor,\\n      except as required for reasonable and customary use in describing the\\n      origin of the Work and reproducing the content of the NOTICE file.\\n\\n   7. Disclaimer of Warranty. Unless required by applicable law or\\n      agreed to in writing, Licensor provides the Work (and each\\n      Contributor provides its Contributions) on an "AS IS" BASIS,\\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\\n      implied, including, without limitation, any warranties or conditions\\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\\n      PARTICULAR PURPOSE. You are solely responsible for determining the\\n      appropriateness of using or redistributing the Work and assume any\\n      risks associated with Your exercise of permissions under this License.\\n\\n   8. Limitation of Liability. In no event and under no legal theory,\\n      whether in tort (including negligence), contract, or otherwise,\\n      unless required by applicable law (such as deliberate and grossly\\n      negligent acts) or agreed to in writing, shall any Contributor be\\n      liable to You for damages, including any direct, indirect, special,\\n      incidental, or consequential damages of any character arising as a\\n      result of this License or out of the use or inability to use the\\n      Work (including but not limited to damages for loss of goodwill,\\n      work stoppage, computer failure or malfunction, or any and all\\n      other commercial damages or losses), even if such Contributor\\n      has been advised of the possibility of such damages.\\n\\n   9. Accepting Warranty or Additional Liability. While redistributing\\n      the Work or Derivative Works thereof, You may choose to offer,\\n      and charge a fee for, acceptance of support, warranty, indemnity,\\n      or other liability obligations and/or rights consistent with this\\n      License. However, in accepting such obligations, You may act only\\n      on Your own behalf and on Your sole responsibility, not on behalf\\n      of any other Contributor, and only if You agree to indemnify,\\n      defend, and hold each Contributor harmless for any liability\\n      incurred by, or claims asserted against, such Contributor by reason\\n      of your accepting any such warranty or additional liability.\\n\\n   END OF TERMS AND CONDITIONS\\n\\n   APPENDIX: How to apply the Apache License to your work.\\n\\n      To apply the Apache License to your work, attach the following\\n      boilerplate notice, with the fields enclosed by brackets "{}"\\n      replaced with your own identifying information. (Don\\\'t include\\n      the brackets!)  The text should be enclosed in the appropriate\\n      comment syntax for the file format. We also recommend that a\\n      file or class name and description of purpose be included on the\\n      same "printed page" as the copyright notice for easier\\n      identification within third-party archives.\\n\\n   Copyright 2025 seed\\n\\n   Licensed under the Apache License, Version 2.0 (the "License");\\n   you may not use this file except in compliance with the License.\\n   You may obtain a copy of the License at\\n\\n       http://www.apache.org/licenses/LICENSE-2.0\\n\\n   Unless required by applicable law or agreed to in writing, software\\n   distributed under the License is distributed on an "AS IS" BASIS,\\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\\n   See the License for the specific language governing permissions and\\n   limitations under the License.\\n\', \'qwen3-4b-APACHE.txt\': \'\\n                                 Apache License\\n                           Version 2.0, January 2004\\n                        http://www.apache.org/licenses/\\n\\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\\n\\n   1. Definitions.\\n\\n      "License" shall mean the terms and conditions for use, reproduction,\\n      and distribution as defined by Sections 1 through 9 of this document.\\n\\n      "Licensor" shall mean the copyright owner or entity authorized by\\n      the copyright owner that is granting the License.\\n\\n      "Legal Entity" shall mean the union of the acting entity and all\\n      other entities that control, are controlled by, or are under common\\n      control with that entity. For the purposes of this definition,\\n      "control" means (i) the power, direct or indirect, to cause the\\n      direction or management of such entity, whether by contract or\\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\\n      outstanding shares, or (iii) beneficial ownership of such entity.\\n\\n      "You" (or "Your") shall mean an individual or Legal Entity\\n      exercising permissions granted by this License.\\n\\n      "Source" form shall mean the preferred form for making modifications,\\n      including but not limited to software source code, documentation\\n      source, and configuration files.\\n\\n      "Object" form shall mean any form resulting from mechanical\\n      transformation or translation of a Source form, including but\\n      not limited to compiled object code, generated documentation,\\n      and conversions to other media types.\\n\\n      "Work" shall mean the work of authorship, whether in Source or\\n      Object form, made available under the License, as indicated by a\\n      copyright notice that is included in or attached to the work\\n      (an example is provided in the Appendix below).\\n\\n      "Derivative Works" shall mean any work, whether in Source or Object\\n      form, that is based on (or derived from) the Work and for which the\\n      editorial revisions, annotations, elaborations, or other modifications\\n      represent, as a whole, an original work of authorship. For the purposes\\n      of this License, Derivative Works shall not include works that remain\\n      separable from, or merely link (or bind by name) to the interfaces of,\\n      the Work and Derivative Works thereof.\\n\\n      "Contribution" shall mean any work of authorship, including\\n      the original version of the Work and any modifications or additions\\n      to that Work or Derivative Works thereof, that is intentionally\\n      submitted to Licensor for inclusion in the Work by the copyright owner\\n      or by an individual or Legal Entity authorized to submit on behalf of\\n      the copyright owner. For the purposes of this definition, "submitted"\\n      means any form of electronic, verbal, or written communication sent\\n      to the Licensor or its representatives, including but not limited to\\n      communication on electronic mailing lists, source code control systems,\\n      and issue tracking systems that are managed by, or on behalf of, the\\n      Licensor for the purpose of discussing and improving the Work, but\\n      excluding communication that is conspicuously marked or otherwise\\n      designated in writing by the copyright owner as "Not a Contribution."\\n\\n      "Contributor" shall mean Licensor and any individual or Legal Entity\\n      on behalf of whom a Contribution has been received by Licensor and\\n      subsequently incorporated within the Work.\\n\\n   2. Grant of Copyright License. Subject to the terms and conditions of\\n      this License, each Contributor hereby grants to You a perpetual,\\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\\n      copyright license to reproduce, prepare Derivative Works of,\\n      publicly display, publicly perform, sublicense, and distribute the\\n      Work and such Derivative Works in Source or Object form.\\n\\n   3. Grant of Patent License. Subject to the terms and conditions of\\n      this License, each Contributor hereby grants to You a perpetual,\\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\\n      (except as stated in this section) patent license to make, have made,\\n      use, offer to sell, sell, import, and otherwise transfer the Work,\\n      where such license applies only to those patent claims licensable\\n      by such Contributor that are necessarily infringed by their\\n      Contribution(s) alone or by combination of their Contribution(s)\\n      with the Work to which such Contribution(s) was submitted. If You\\n      institute patent litigation against any entity (including a\\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\\n      or a Contribution incorporated within the Work constitutes direct\\n      or contributory patent infringement, then any patent licenses\\n      granted to You under this License for that Work shall terminate\\n      as of the date such litigation is filed.\\n\\n   4. Redistribution. You may reproduce and distribute copies of the\\n      Work or Derivative Works thereof in any medium, with or without\\n      modifications, and in Source or Object form, provided that You\\n      meet the following conditions:\\n\\n      (a) You must give any other recipients of the Work or\\n          Derivative Works a copy of this License; and\\n\\n      (b) You must cause any modified files to carry prominent notices\\n          stating that You changed the files; and\\n\\n      (c) You must retain, in the Source form of any Derivative Works\\n          that You distribute, all copyright, patent, trademark, and\\n          attribution notices from the Source form of the Work,\\n          excluding those notices that do not pertain to any part of\\n          the Derivative Works; and\\n\\n      (d) If the Work includes a "NOTICE" text file as part of its\\n          distribution, then any Derivative Works that You distribute must\\n          include a readable copy of the attribution notices contained\\n          within such NOTICE file, excluding those notices that do not\\n          pertain to any part of the Derivative Works, in at least one\\n          of the following places: within a NOTICE text file distributed\\n          as part of the Derivative Works; within the Source form or\\n          documentation, if provided along with the Derivative Works; or,\\n          within a display generated by the Derivative Works, if and\\n          wherever such third-party notices normally appear. The contents\\n          of the NOTICE file are for informational purposes only and\\n          do not modify the License. You may add Your own attribution\\n          notices within Derivative Works that You distribute, alongside\\n          or as an addendum to the NOTICE text from the Work, provided\\n          that such additional attribution notices cannot be construed\\n          as modifying the License.\\n\\n      You may add Your own copyright statement to Your modifications and\\n      may provide additional or different license terms and conditions\\n      for use, reproduction, or distribution of Your modifications, or\\n      for any such Derivative Works as a whole, provided Your use,\\n      reproduction, and distribution of the Work otherwise complies with\\n      the conditions stated in this License.\\n\\n   5. Submission of Contributions. Unless You explicitly state otherwise,\\n      any Contribution intentionally submitted for inclusion in the Work\\n      by You to the Licensor shall be under the terms and conditions of\\n      this License, without any additional terms or conditions.\\n      Notwithstanding the above, nothing herein shall supersede or modify\\n      the terms of any separate license agreement you may have executed\\n      with Licensor regarding such Contributions.\\n\\n   6. Trademarks. This License does not grant permission to use the trade\\n      names, trademarks, service marks, or product names of the Licensor,\\n      except as required for reasonable and customary use in describing the\\n      origin of the Work and reproducing the content of the NOTICE file.\\n\\n   7. Disclaimer of Warranty. Unless required by applicable law or\\n      agreed to in writing, Licensor provides the Work (and each\\n      Contributor provides its Contributions) on an "AS IS" BASIS,\\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\\n      implied, including, without limitation, any warranties or conditions\\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\\n      PARTICULAR PURPOSE. You are solely responsible for determining the\\n      appropriateness of using or redistributing the Work and assume any\\n      risks associated with Your exercise of permissions under this License.\\n\\n   8. Limitation of Liability. In no event and under no legal theory,\\n      whether in tort (including negligence), contract, or otherwise,\\n      unless required by applicable law (such as deliberate and grossly\\n      negligent acts) or agreed to in writing, shall any Contributor be\\n      liable to You for damages, including any direct, indirect, special,\\n      incidental, or consequential damages of any character arising as a\\n      result of this License or out of the use or inability to use the\\n      Work (including but not limited to damages for loss of goodwill,\\n      work stoppage, computer failure or malfunction, or any and all\\n      other commercial damages or losses), even if such Contributor\\n      has been advised of the possibility of such damages.\\n\\n   9. Accepting Warranty or Additional Liability. While redistributing\\n      the Work or Derivative Works thereof, You may choose to offer,\\n      and charge a fee for, acceptance of support, warranty, indemnity,\\n      or other liability obligations and/or rights consistent with this\\n      License. However, in accepting such obligations, You may act only\\n      on Your own behalf and on Your sole responsibility, not on behalf\\n      of any other Contributor, and only if You agree to indemnify,\\n      defend, and hold each Contributor harmless for any liability\\n      incurred by, or claims asserted against, such Contributor by reason\\n      of your accepting any such warranty or additional liability.\\n\\n   END OF TERMS AND CONDITIONS\\n\\n   APPENDIX: How to apply the Apache License to your work.\\n\\n      To apply the Apache License to your work, attach the following\\n      boilerplate notice, with the fields enclosed by brackets "[]"\\n      replaced with your own identifying information. (Don\\\'t include\\n      the brackets!)  The text should be enclosed in the appropriate\\n      comment syntax for the file format. We also recommend that a\\n      file or class name and description of purpose be included on the\\n      same "printed page" as the copyright notice for easier\\n      identification within third-party archives.\\n\\n   Copyright 2024 Alibaba Cloud\\n\\n   Licensed under the Apache License, Version 2.0 (the "License");\\n   you may not use this file except in compliance with the License.\\n   You may obtain a copy of the License at\\n\\n       http://www.apache.org/licenses/LICENSE-2.0\\n\\n   Unless required by applicable law or agreed to in writing, software\\n   distributed under the License is distributed on an "AS IS" BASIS,\\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\\n   See the License for the specific language governing permissions and\\n   limitations under the License.\', \'bfs-klein-guide.md\': \'Source: https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/blob/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/docs/flux-2-klein.md\\n\\n# Flux 2 Klein Head-Swap LoRAs\\n\\nThese BFS weights target Flux 2 Klein 4B/9B variants. They use the body-first input contract:\\n\\n- **Image 1:** target body/base image\\n- **Image 2:** reference face/head\\n\\nDo not reverse the images for these releases.\\n\\n## Downloads\\n\\n- [`bfs_head_v1_flux-klein_4b.safetensors`](../bfs_head_v1_flux-klein_4b.safetensors)\\n- [`bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors`](../bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors)\\n- [`bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors`](../bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors)\\n- [`bfs_head_v1.1_optional_flux-klein_4b.safetensors`](../bfs_head_v1.1_optional_flux-klein_4b.safetensors)\\n\\n## Prompt\\n\\n```text\\nhead_swap: start with Picture 1 as the base image, keeping its lighting, environment, and background. remove the head from Picture 1 completely and replace it with the head from Picture 2, strictly preserving the hair, eye color, and nose structure of Picture 2. copy the direction of the eye, head rotation, and micro-expressions from Picture 1. Keep the body, clothing, pose, framing, and background intact.\\n```\\n\\nYou can add a short expression or pose instruction when it is visible in the target image. Keep\\nthe edit focused on the head and avoid requesting an unrelated scene change in the same prompt.\\n\\n## Workflow\\n\\n[`workflows/Head Swap V1 Flux 2 Klein 4b_9b (base_distill).json`](../workflows/Head%20Swap%20V1%20Flux%202%20Klein%204b_9b%20(base_distill).json)\\n\\nStart around LoRA strength `1.0`; adjust after checking identity preservation, hair boundaries,\\nexpression transfer, and skin blending.\\n\\n[Back to the repository index](../README.md)\\n\', \'mediapipe-source-APACHE.txt\': \'                                 Apache License\\n                           Version 2.0, January 2004\\n                        http://www.apache.org/licenses/\\n\\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\\n\\n   1. Definitions.\\n\\n      "License" shall mean the terms and conditions for use, reproduction,\\n      and distribution as defined by Sections 1 through 9 of this document.\\n\\n      "Licensor" shall mean the copyright owner or entity authorized by\\n      the copyright owner that is granting the License.\\n\\n      "Legal Entity" shall mean the union of the acting entity and all\\n      other entities that control, are controlled by, or are under common\\n      control with that entity. For the purposes of this definition,\\n      "control" means (i) the power, direct or indirect, to cause the\\n      direction or management of such entity, whether by contract or\\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\\n      outstanding shares, or (iii) beneficial ownership of such entity.\\n\\n      "You" (or "Your") shall mean an individual or Legal Entity\\n      exercising permissions granted by this License.\\n\\n      "Source" form shall mean the preferred form for making modifications,\\n      including but not limited to software source code, documentation\\n      source, and configuration files.\\n\\n      "Object" form shall mean any form resulting from mechanical\\n      transformation or translation of a Source form, including but\\n      not limited to compiled object code, generated documentation,\\n      and conversions to other media types.\\n\\n      "Work" shall mean the work of authorship, whether in Source or\\n      Object form, made available under the License, as indicated by a\\n      copyright notice that is included in or attached to the work\\n      (an example is provided in the Appendix below).\\n\\n      "Derivative Works" shall mean any work, whether in Source or Object\\n      form, that is based on (or derived from) the Work and for which the\\n      editorial revisions, annotations, elaborations, or other modifications\\n      represent, as a whole, an original work of authorship. For the purposes\\n      of this License, Derivative Works shall not include works that remain\\n      separable from, or merely link (or bind by name) to the interfaces of,\\n      the Work and Derivative Works thereof.\\n\\n      "Contribution" shall mean any work of authorship, including\\n      the original version of the Work and any modifications or additions\\n      to that Work or Derivative Works thereof, that is intentionally\\n      submitted to Licensor for inclusion in the Work by the copyright owner\\n      or by an individual or Legal Entity authorized to submit on behalf of\\n      the copyright owner. For the purposes of this definition, "submitted"\\n      means any form of electronic, verbal, or written communication sent\\n      to the Licensor or its representatives, including but not limited to\\n      communication on electronic mailing lists, source code control systems,\\n      and issue tracking systems that are managed by, or on behalf of, the\\n      Licensor for the purpose of discussing and improving the Work, but\\n      excluding communication that is conspicuously marked or otherwise\\n      designated in writing by the copyright owner as "Not a Contribution."\\n\\n      "Contributor" shall mean Licensor and any individual or Legal Entity\\n      on behalf of whom a Contribution has been received by Licensor and\\n      subsequently incorporated within the Work.\\n\\n   2. Grant of Copyright License. Subject to the terms and conditions of\\n      this License, each Contributor hereby grants to You a perpetual,\\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\\n      copyright license to reproduce, prepare Derivative Works of,\\n      publicly display, publicly perform, sublicense, and distribute the\\n      Work and such Derivative Works in Source or Object form.\\n\\n   3. Grant of Patent License. Subject to the terms and conditions of\\n      this License, each Contributor hereby grants to You a perpetual,\\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\\n      (except as stated in this section) patent license to make, have made,\\n      use, offer to sell, sell, import, and otherwise transfer the Work,\\n      where such license applies only to those patent claims licensable\\n      by such Contributor that are necessarily infringed by their\\n      Contribution(s) alone or by combination of their Contribution(s)\\n      with the Work to which such Contribution(s) was submitted. If You\\n      institute patent litigation against any entity (including a\\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\\n      or a Contribution incorporated within the Work constitutes direct\\n      or contributory patent infringement, then any patent licenses\\n      granted to You under this License for that Work shall terminate\\n      as of the date such litigation is filed.\\n\\n   4. Redistribution. You may reproduce and distribute copies of the\\n      Work or Derivative Works thereof in any medium, with or without\\n      modifications, and in Source or Object form, provided that You\\n      meet the following conditions:\\n\\n      (a) You must give any other recipients of the Work or\\n          Derivative Works a copy of this License; and\\n\\n      (b) You must cause any modified files to carry prominent notices\\n          stating that You changed the files; and\\n\\n      (c) You must retain, in the Source form of any Derivative Works\\n          that You distribute, all copyright, patent, trademark, and\\n          attribution notices from the Source form of the Work,\\n          excluding those notices that do not pertain to any part of\\n          the Derivative Works; and\\n\\n      (d) If the Work includes a "NOTICE" text file as part of its\\n          distribution, then any Derivative Works that You distribute must\\n          include a readable copy of the attribution notices contained\\n          within such NOTICE file, excluding those notices that do not\\n          pertain to any part of the Derivative Works, in at least one\\n          of the following places: within a NOTICE text file distributed\\n          as part of the Derivative Works; within the Source form or\\n          documentation, if provided along with the Derivative Works; or,\\n          within a display generated by the Derivative Works, if and\\n          wherever such third-party notices normally appear. The contents\\n          of the NOTICE file are for informational purposes only and\\n          do not modify the License. You may add Your own attribution\\n          notices within Derivative Works that You distribute, alongside\\n          or as an addendum to the NOTICE text from the Work, provided\\n          that such additional attribution notices cannot be construed\\n          as modifying the License.\\n\\n      You may add Your own copyright statement to Your modifications and\\n      may provide additional or different license terms and conditions\\n      for use, reproduction, or distribution of Your modifications, or\\n      for any such Derivative Works as a whole, provided Your use,\\n      reproduction, and distribution of the Work otherwise complies with\\n      the conditions stated in this License.\\n\\n   5. Submission of Contributions. Unless You explicitly state otherwise,\\n      any Contribution intentionally submitted for inclusion in the Work\\n      by You to the Licensor shall be under the terms and conditions of\\n      this License, without any additional terms or conditions.\\n      Notwithstanding the above, nothing herein shall supersede or modify\\n      the terms of any separate license agreement you may have executed\\n      with Licensor regarding such Contributions.\\n\\n   6. Trademarks. This License does not grant permission to use the trade\\n      names, trademarks, service marks, or product names of the Licensor,\\n      except as required for reasonable and customary use in describing the\\n      origin of the Work and reproducing the content of the NOTICE file.\\n\\n   7. Disclaimer of Warranty. Unless required by applicable law or\\n      agreed to in writing, Licensor provides the Work (and each\\n      Contributor provides its Contributions) on an "AS IS" BASIS,\\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\\n      implied, including, without limitation, any warranties or conditions\\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\\n      PARTICULAR PURPOSE. You are solely responsible for determining the\\n      appropriateness of using or redistributing the Work and assume any\\n      risks associated with Your exercise of permissions under this License.\\n\\n   8. Limitation of Liability. In no event and under no legal theory,\\n      whether in tort (including negligence), contract, or otherwise,\\n      unless required by applicable law (such as deliberate and grossly\\n      negligent acts) or agreed to in writing, shall any Contributor be\\n      liable to You for damages, including any direct, indirect, special,\\n      incidental, or consequential damages of any character arising as a\\n      result of this License or out of the use or inability to use the\\n      Work (including but not limited to damages for loss of goodwill,\\n      work stoppage, computer failure or malfunction, or any and all\\n      other commercial damages or losses), even if such Contributor\\n      has been advised of the possibility of such damages.\\n\\n   9. Accepting Warranty or Additional Liability. While redistributing\\n      the Work or Derivative Works thereof, You may choose to offer,\\n      and charge a fee for, acceptance of support, warranty, indemnity,\\n      or other liability obligations and/or rights consistent with this\\n      License. However, in accepting such obligations, You may act only\\n      on Your own behalf and on Your sole responsibility, not on behalf\\n      of any other Contributor, and only if You agree to indemnify,\\n      defend, and hold each Contributor harmless for any liability\\n      incurred by, or claims asserted against, such Contributor by reason\\n      of your accepting any such warranty or additional liability.\\n\\n   END OF TERMS AND CONDITIONS\\n\\n   APPENDIX: How to apply the Apache License to your work.\\n\\n      To apply the Apache License to your work, attach the following\\n      boilerplate notice, with the fields enclosed by brackets "[]"\\n      replaced with your own identifying information. (Don\\\'t include\\n      the brackets!)  The text should be enclosed in the appropriate\\n      comment syntax for the file format. We also recommend that a\\n      file or class name and description of purpose be included on the\\n      same "printed page" as the copyright notice for easier\\n      identification within third-party archives.\\n\\n   Copyright [yyyy] [name of copyright owner]\\n\\n   Licensed under the Apache License, Version 2.0 (the "License");\\n   you may not use this file except in compliance with the License.\\n   You may obtain a copy of the License at\\n\\n       http://www.apache.org/licenses/LICENSE-2.0\\n\\n   Unless required by applicable law or agreed to in writing, software\\n   distributed under the License is distributed on an "AS IS" BASIS,\\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\\n   See the License for the specific language governing permissions and\\n   limitations under the License.\\n\\n===========================================================================\\nFor files under tasks/cc/text/language_detector/custom_ops/utils/utf/\\n===========================================================================\\n/*\\n * The authors of this software are Rob Pike and Ken Thompson.\\n *              Copyright (c) 2002 by Lucent Technologies.\\n * Permission to use, copy, modify, and distribute this software for any\\n * purpose without fee is hereby granted, provided that this entire notice\\n * is included in all copies of any software which is or includes a copy\\n * or modification of this software and in all copies of the supporting\\n * documentation for such software.\\n * THIS SOFTWARE IS BEING PROVIDED "AS IS", WITHOUT ANY EXPRESS OR IMPLIED\\n * WARRANTY.  IN PARTICULAR, NEITHER THE AUTHORS NOR LUCENT TECHNOLOGIES MAKE ANY\\n * REPRESENTATION OR WARRANTY OF ANY KIND CONCERNING THE MERCHANTABILITY\\n * OF THIS SOFTWARE OR ITS FITNESS FOR ANY PARTICULAR PURPOSE.\\n */\\n\'}\nif not LICENSE_FILES and \'__file__\' in globals():\n _license_root=Path(__file__).with_name(\'licenses\')\n if _license_root.is_dir():LICENSE_FILES={p.name:p.read_text() for p in _license_root.iterdir() if p.is_file()}\n\ndef file_sha(path):\n h=hashlib.sha256()\n with open(path,\'rb\') as f:\n  for chunk in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(chunk)\n return h.hexdigest()\n\ndef selected_assets(pack):\n if pack not in (\'create_edit\',\'finish\'):raise ValueError(\'Use create_edit or finish\')\n return [(a[\'folder\'],a[\'name\'],a[\'source\'],a[\'sha256\'],a[\'bytes\']) for a in CATALOG[\'assets\'] if a[\'pack\']==pack]\n\ndef validate_manifest(manifest):\n if manifest.get(\'profile\')!=\'commercial-v7\' or not manifest.get(\'complete\'):raise RuntimeError(\'Not a complete commercial-v7 pack\')\n expected=selected_assets(manifest.get(\'pack\'))\n records=manifest.get(\'files\',[])\n wanted={(n,z,d) for _,n,_,d,z in expected}\n actual={(a.get(\'path\'),a.get(\'bytes\'),a.get(\'sha256\')) for a in records}\n if wanted!=actual or len(records)!=len(wanted):raise RuntimeError(\'Manifest is not the approved commercial model allowlist\')\n return True\n\ndef reject_extra_models(root,pack):\n allowed={a[1] for a in selected_assets(pack)}\n for p in Path(root).rglob(\'*\'):\n  if p.suffix.lower() in (\'.onnx\',\'.zip\',\'.pth\',\'.pt\',\'.safetensors\',\'.task\',\'.bin\') and (p.parent!=Path(root) or p.name not in allowed):raise RuntimeError(\'Unapproved extra model in pack: \'+p.name)\n\ndef transfer(url,target,digest,size):\n target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)\n if target.exists():\n  if target.stat().st_size==size and file_sha(target)==digest:\n   print(\'Verified; reused:\',target.name,flush=True);return\n  raise RuntimeError(\'Existing file failed verification; preserve or move it before retrying: \'+str(target))\n partial=target.with_name(target.name+\'.partial\')\n for attempt in range(3):\n  offset=partial.stat().st_size if partial.exists() else 0\n  if offset>=size:\n   if offset==size and file_sha(partial)==digest:partial.replace(target);return\n   partial.unlink();offset=0\n  if shutil.disk_usage(target.parent).free<size-offset+512*1024**2:raise RuntimeError(\'Insufficient free disk for \'+target.name)\n  req=urllib.request.Request(url,headers={\'Range\':f\'bytes={offset}-\'} if offset else {})\n  try:\n   with urllib.request.urlopen(req,timeout=120) as response:\n    resume=offset>0 and response.status==206\n    if resume and not response.headers.get(\'Content-Range\',\'\').startswith(f\'bytes {offset}-\'):raise RuntimeError(\'Unexpected range response\')\n    written=offset if resume else 0;reported=written\n    with open(partial,\'ab\' if resume else \'wb\') as f:\n     while True:\n      data=response.read(8*1024*1024)\n      if not data:break\n      f.write(data);written+=len(data)\n      if written-reported>=128*1024**2:\n       print(f\'{target.name}: {written/size:.0%}\',flush=True);reported=written\n   if partial.stat().st_size!=size or file_sha(partial)!=digest:raise RuntimeError(\'Incomplete download or checksum mismatch\')\n   partial.replace(target);return\n  except (OSError,urllib.error.URLError,RuntimeError) as e:\n   if attempt==2:raise RuntimeError(\'Download failed for \'+target.name+\'; rerun to retry its partial file.\') from e\n   print(\'Retrying\',target.name,type(e).__name__,flush=True);time.sleep(2)\n\ndef build_pack(pack,output_root=Path(\'/kaggle/working/image-studio-model-packs\'),input_root=Path(\'/kaggle/input\')):\n assets=selected_assets(pack)\n root=Path(output_root)/pack;root.mkdir(parents=True,exist_ok=True)\n lock=open(root/\'.builder.lock\',\'w\')\n try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\n except BlockingIOError:lock.close();raise RuntimeError(\'This pack is already being built.\')\n try:\n  manifest=root/\'manifest.json\'\n  # An interrupted rebuild must never leave a completed manifest behind.\n  if manifest.exists():manifest.replace(root/\'manifest.previous.json\')\n  print(\'Building\',pack,\'on CPU; model bytes:\',sum(a[4] for a in assets),flush=True)\n  reject_extra_models(root,pack)\n  records=[]\n  for folder,name,url,digest,size in assets:\n   target=root/name;target.parent.mkdir(parents=True,exist_ok=True)\n   # Saved packs must contain real files: input symlinks would not be a durable output.\n   if not target.exists() and Path(input_root).exists():\n    for candidate in Path(input_root).rglob(name):\n     if candidate.is_file() and candidate.stat().st_size==size and file_sha(candidate)==digest:\n      if shutil.disk_usage(target.parent).free<size+512*1024**2:raise RuntimeError(\'Insufficient disk to copy saved model\')\n      tmp=target.with_name(name+\'.partial\');shutil.copyfile(candidate,tmp);tmp.replace(target);print(\'Copied saved input:\',name,flush=True);break\n   transfer(url,target,digest,size)\n   if target.is_symlink() or target.stat().st_size!=size or file_sha(target)!=digest:raise RuntimeError(\'Pack verification failed: \'+name)\n   records.append({\'path\':target.relative_to(root).as_posix(),\'bytes\':size,\'sha256\':digest,\'source\':url})\n  data={\'schema\':2,\'profile\':\'commercial-v7\',\'pack\':pack,\'complete\':True,\'model_bytes\':sum(a[4] for a in assets),\'files\':records}\n  validate_manifest(data)\n  data[\'license_catalog\']=CATALOG\n  for name,content in LICENSE_FILES.items():(root/name).write_text(content)\n  tmp=manifest.with_suffix(\'.json.partial\');tmp.write_text(json.dumps(data,indent=2));tmp.replace(manifest)\n  print(\'\\nPACK COMPLETE:\',root,flush=True)\n  print(\'Save Version with outputs. In Image Studio Commercial v7: Add Input → Notebook Output Files → select this saved version.\',flush=True)\n  print(\'No GPU or image generation was used. Keep model packs private and follow model licenses.\',flush=True)\n  return data\n finally:fcntl.flock(lock,fcntl.LOCK_UN);lock.close()\n\ndef dataset_identity(owner,manifest):\n import re\n if not isinstance(owner,str) or not re.fullmatch(r\'[A-Za-z0-9_-]+\',owner):raise ValueError(\'Set DATASET_OWNER to your Kaggle username or use configured Kaggle credentials.\')\n fingerprint=hashlib.sha256(json.dumps(manifest[\'files\'],sort_keys=True).encode()).hexdigest()[:12]\n # Local pack names may have underscores; Kaggle dataset URL slugs may not.\n pack_slug=re.sub(r\'[^a-z0-9]+\',\'-\',manifest[\'pack\'].lower()).strip(\'-\')\n slug=\'image-studio-commercial-\'+pack_slug+\'-\'+fingerprint\n if not 6<=len(slug)<=50 or not re.fullmatch(r\'[a-z0-9]+(?:-[a-z0-9]+)*\',slug):raise ValueError(\'Invalid Kaggle Dataset slug; no files were uploaded.\')\n return owner+\'/\'+slug,fingerprint\n\ndef kaggle_error_text(error):\n """Keep the actionable reason without exposing credentials or response headers."""\n import re\n message=str(error)\n response=getattr(error,\'response\',None)\n if response is not None:\n  try:\n   data=response.json()\n   if isinstance(data,dict):message=str(data.get(\'message\') or data.get(\'error\') or data.get(\'detail\') or message)\n  except (ValueError,TypeError):pass\n for key in (\'KAGGLE_API_TOKEN\',\'KAGGLE_KEY\'):\n  value=os.environ.get(key)\n  if value:message=message.replace(value,\'[REDACTED]\')\n message=re.sub(r\'(?i)Bearer\\s+[^\\s,;]+\',\'Bearer [REDACTED]\',message)\n message=re.sub(r\'(?i)((?:api[_-]?key|access_token|token|authorization)\\s*[=:]\\s*)[^\\s,;]+\',r\'\\1[REDACTED]\',message)\n return message[:1800]\n\ndef save_private_dataset(pack_root,owner=\'\',api=None):\n root=Path(pack_root);manifest=json.loads((root/\'manifest.json\').read_text())\n validate_manifest(manifest)\n reject_extra_models(root,manifest[\'pack\'])\n for name in LICENSE_FILES:\n  if not (root/name).is_file() or (root/name).read_text()!=LICENSE_FILES[name]:raise RuntimeError(\'Missing/changed license notice: \'+name)\n for record in manifest[\'files\']:\n  p=root/record[\'path\']\n  if p.is_symlink() or not p.is_file() or p.stat().st_size!=record[\'bytes\'] or file_sha(p)!=record[\'sha256\']:raise RuntimeError(\'Model verification failed before Dataset upload: \'+p.name)\n if api is None:\n  # Kaggle API uses private notebook credentials. Never print or store their values.\n  if not os.environ.get(\'KAGGLE_API_TOKEN\') and not os.environ.get(\'KAGGLE_KEY\'):\n   try:\n    from kaggle_secrets import UserSecretsClient\n    os.environ[\'KAGGLE_API_TOKEN\']=UserSecretsClient().get_secret(\'KAGGLE_API_TOKEN\')\n   except Exception:pass\n  if not os.environ.get(\'KAGGLE_API_TOKEN\') and not os.environ.get(\'KAGGLE_KEY\'):\n   try:\n    from kaggle_secrets import UserSecretsClient\n    secrets=UserSecretsClient()\n    os.environ[\'KAGGLE_USERNAME\']=secrets.get_secret(\'KAGGLE_USERNAME\')\n    os.environ[\'KAGGLE_KEY\']=secrets.get_secret(\'KAGGLE_KEY\')\n   except Exception:pass\n  from kaggle.api.kaggle_api_extended import KaggleApi\n  api=KaggleApi();api.authenticate()\n owner=owner or api.config_values.get(\'username\',\'\')\n handle,fingerprint=dataset_identity(owner,manifest)\n url=\'https://www.kaggle.com/datasets/\'+handle\n receipt=root/\'dataset-upload.json\'\n if receipt.exists():\n  previous=json.loads(receipt.read_text())\n  if previous.get(\'handle\')==handle and previous.get(\'submitted\') and previous.get(\'manifest_fingerprint\')==fingerprint:\n   print(\'Previously submitted Dataset:\',url,flush=True);return url\n # Match the previously working builder: call create_new directly.\n # Do not gate creation on our own dataset_status/datasets.get request.\n # Kaggle can deny lookup on a not-yet-created/private Dataset; that does not\n # establish that creation is forbidden. create_new never versions/overwrites.\n metadata={\'title\':\'Image Studio \'+manifest[\'pack\']+\' \'+fingerprint,\'id\':handle,\'licenses\':[{\'name\':\'other\'}],\'description\':\'Private verified model pack for Image Studio. Original model licenses apply; no relicensing. See manifest.json for model sources and SHA256 values.\'}\n (root/\'dataset-metadata.json\').write_text(json.dumps(metadata,indent=2))\n # Stage hardlinks to the exact pack files, avoiding an extra multi-GB copy.\n # Do not upload builder locks, old manifests, receipts or failure diagnostics.\n import tempfile\n with tempfile.TemporaryDirectory(prefix=\'.studio-upload-\',dir=root.parent) as tmp:\n  stage=Path(tmp)\n  for name in [r[\'path\'] for r in manifest[\'files\']]+list(LICENSE_FILES)+[\'manifest.json\',\'dataset-metadata.json\']:\n   os.link(root/name,stage/name)\n  result=api.dataset_create_new(str(stage),public=False,quiet=False,convert_to_csv=False,dir_mode=\'skip\')\n field=lambda key:result.get(key) if isinstance(result,dict) else getattr(result,key,None)\n error=field(\'error\') or field(\'error_message\')\n status=field(\'status\')\n if error or (status and str(status).lower() in (\'error\',\'failed\',\'failure\')):\n  raise RuntimeError(\'Kaggle rejected Dataset creation: \'+kaggle_error_text(error or status)+\'. Verified models are retained; no model downloads are needed.\')\n if result is None:raise RuntimeError(\'Kaggle returned no creation response. Verified models are retained. Check the Dataset page before retrying.\')\n receipt.write_text(json.dumps({\'handle\':handle,\'url\':url,\'submitted\':True,\'requested_private\':True,\'manifest_fingerprint\':fingerprint},indent=2))\n print(\'PRIVATE DATASET SUBMITTED:\',url,flush=True)\n print(\'Wait for Kaggle processing, then Image Studio Commercial v7 → Add Input → Your Datasets. No PC download needed.\',flush=True)\n return url\n\n\ndef publish_pack(pack_root,owner=\'\',api=None):\n try:return save_private_dataset(pack_root,owner,api)\n except (Exception,SystemExit) as error:\n  detail=kaggle_error_text(error)\n  root=Path(pack_root)\n  (root/\'dataset-error.json\').write_text(json.dumps({\'stage\':\'dataset-publication\',\'reason\':detail,\'models_retained\':True},indent=2))\n  raise RuntimeError(\'Dataset publication failed: \'+detail+\' Save Version with outputs before closing this session. Rerunning reuses verified weights. Diagnostic: \'+str(root/\'dataset-error.json\')) from None\n\nif __name__==\'__main__\':\n build_pack(PACK)\n if AUTO_CREATE_DATASET:publish_pack(Path(\'/kaggle/working/image-studio-model-packs\')/PACK,DATASET_OWNER)\n'
FACE_NODE_SOURCE = '"""CPU landmark mesh compositor. No identity encoder or learned face-swap weights."""\nimport os,json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport cv2\nimport folder_paths\n\ndef rgb(t):\n if t.ndim!=4 or t.shape[0]!=1:raise ValueError(\'Use one image per input.\')\n return (t[0].cpu().numpy().clip(0,1)*255).round().astype(np.uint8)\ndef detect(image):\n import mediapipe as mp\n task=Path(os.environ.get(\'STUDIO_LANDMARK_MODEL\',str(Path(folder_paths.models_dir)/\'mediapipe/face_landmarker.task\')))\n if not task.is_file():raise ValueError(\'Attach the commercial finish pack: face_landmarker.task is missing.\')\n options=mp.tasks.vision.FaceLandmarkerOptions(base_options=mp.tasks.BaseOptions(model_asset_path=str(task)),num_faces=10,output_face_blendshapes=False,output_facial_transformation_matrixes=False)\n with mp.tasks.vision.FaceLandmarker.create_from_options(options) as model:\n  result=model.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=np.ascontiguousarray(image)))\n h,w=image.shape[:2]\n points=[np.array([(min(w-1,max(0,p.x*w)),min(h-1,max(0,p.y*h))) for p in face[:468]],np.float32) for face in result.face_landmarks]\n return sorted(points,key=lambda p:p[:,0].mean())\ndef composite(source,target,source_index=0,target_index=0,blend_mode="Feather"):\n sp,tp=detect(source),detect(target)\n if not 0<=source_index<len(sp) or not 0<=target_index<len(tp):raise ValueError(f\'Faces found: source={len(sp)}, target={len(tp)}; indices start at 0, left to right.\')\n a,b=sp[source_index],tp[target_index];h,w=target.shape[:2];warp=np.zeros_like(target)\n # Triangulation derives only from landmark positions; never loads an embedding model.\n hull=cv2.convexHull(b).reshape(-1,2);sub=cv2.Subdiv2D((0,0,w,h))\n for p in b:sub.insert(tuple(map(float,p)))\n for tri in sub.getTriangleList():\n  pts=np.array(tri).reshape(3,2)\n  if np.any(pts[:,0]<0) or np.any(pts[:,0]>=w) or np.any(pts[:,1]<0) or np.any(pts[:,1]>=h):continue\n  ids=[int(np.argmin(np.linalg.norm(b-p,axis=1))) for p in pts]\n  if len(set(ids))<3:continue\n  src=a[ids];dst=b[ids];sx,sy,sw,sh=cv2.boundingRect(src);x,y,tw,th=cv2.boundingRect(dst)\n  if min(sw,sh,tw,th)<1:continue\n  mat=cv2.getAffineTransform((src-[sx,sy]).astype(np.float32),(dst-[x,y]).astype(np.float32))\n  tile=cv2.warpAffine(source[sy:sy+sh,sx:sx+sw],mat,(tw,th),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT_101)\n  mask=np.zeros((th,tw),np.uint8);cv2.fillConvexPoly(mask,np.int32(dst-[x,y]),255)\n  view=warp[y:y+th,x:x+tw];view[mask>0]=tile[mask>0]\n mask=np.zeros((h,w),np.uint8);cv2.fillConvexPoly(mask,np.int32(hull),255)\n # Keep edge inside the face hull and use Poisson blending for lighting continuity.\n mask=cv2.erode(mask,np.ones((3,3),np.uint8))\n x,y,bw,bh=cv2.boundingRect(mask)\n if not np.any(mask):raise ValueError(\'Face is too small to composite.\')\n if blend_mode==\'Poisson\':\n  out=cv2.seamlessClone(warp,target,mask.copy(),(x+bw//2,y+bh//2),cv2.NORMAL_CLONE)\n else:\n  # Preserve the transferred features; Poisson blending can weaken identity contrast.\n  alpha=cv2.distanceTransform(mask,cv2.DIST_L2,3);alpha=np.clip(alpha/max(2,min(bw,bh)*0.04),0,1)[...,None]\n  out=(warp*alpha+target*(1-alpha)).round().astype(np.uint8)\n out[mask==0]=target[mask==0]\n return out,dict(method=\'mediapipe-mesh-pixel-composite\',source_faces=len(sp),target_faces=len(tp),source_index=source_index,target_index=target_index,blend_mode=blend_mode,limitations=\'Use similar pose/expression; this does not synthesize an identity or guarantee likeness.\')\nclass CommercialMeshSwap:\n @classmethod\n def INPUT_TYPES(cls):return {\'required\':{\'source\':(\'IMAGE\',),\'target\':(\'IMAGE\',),\'source_index\':(\'INT\',{\'default\':0,\'min\':0,\'max\':9}),\'target_index\':(\'INT\',{\'default\':0,\'min\':0,\'max\':9}),\'blend_mode\':([\'Feather\',\'Poisson\'],)}}\n RETURN_TYPES=(\'IMAGE\',\'STRING\');RETURN_NAMES=(\'image\',\'report\');FUNCTION=\'swap\';CATEGORY=\'Image Studio Commercial/Face\'\n def swap(self,source,target,source_index,target_index,blend_mode="Feather"):\n  out,report=composite(rgb(source),rgb(target),source_index,target_index,blend_mode)\n  return torch.from_numpy(out.astype(np.float32)/255).unsqueeze(0),json.dumps(report)\nclass CommercialMaskedEdit:\n @classmethod\n def INPUT_TYPES(cls):return {\'required\':{\'original\':(\'IMAGE\',),\'edited\':(\'IMAGE\',),\'mask\':(\'MASK\',),\'feather\':(\'INT\',{\'default\':12,\'min\':0,\'max\':128})}}\n RETURN_TYPES=(\'IMAGE\',);FUNCTION=\'paste\';CATEGORY=\'Image Studio Commercial/Edit\'\n def paste(self,original,edited,mask,feather):\n  a,b=rgb(original),rgb(edited);h,w=a.shape[:2];b=cv2.resize(b,(w,h),interpolation=cv2.INTER_LANCZOS4)\n  m=mask[0].cpu().numpy();m=cv2.resize(m,(w,h));m=np.clip(m,0,1)\n  if not m.any():raise ValueError(\'Paint a mask in LoadImage → MaskEditor first (white = replace).\')\n  if feather:m=cv2.GaussianBlur(m,(0,0),feather/3)\n  out=a*(1-m[...,None])+b*m[...,None]\n  return (torch.from_numpy(out.astype(np.float32)/255).unsqueeze(0),)\nNODE_CLASS_MAPPINGS={c.__name__:c for c in [CommercialMeshSwap,CommercialMaskedEdit]}\n'
JJP_NODE_SOURCE = '"""JJP brief, approved cast and deterministic artwork export stages."""\nfrom pathlib import Path\nimport json,hashlib,re,uuid,os,datetime\nimport numpy as np\nimport torch\nfrom PIL import Image,ImageDraw,ImageFont,ImageOps,PngImagePlugin\nimport folder_paths\nPRESETS=json.loads((Path(__file__).parent/\'jjp_presets.json\').read_text())\nCAST_ROLES=[\'Lead One\',\'Lead Two\',\'Accuser (non-human)\',\'Seeker (faceless)\']\n\ndef slug(value):return re.sub(r\'[^a-zA-Z0-9_-]+\',\'-\',value).strip(\'-\')[:70] or \'untitled\'\ndef output_root():return Path(folder_paths.get_output_directory())/\'JJP\'\ndef cast_roots():\n roots=[output_root()/\'cast\']\n inp=Path(os.environ.get(\'JJP_INPUT_ROOT\',\'/kaggle/input\'))\n if inp.exists():roots.append(inp)\n return roots\n\ndef cast_records():\n records={}\n for root in cast_roots():\n  if not root.exists():continue\n  for path in root.rglob(\'*.cast.json\'):\n   try:\n    record=json.loads(path.read_text());image=path.parent/record[\'image\']\n    if not record.get(\'approved\') or Path(record[\'image\']).name!=record[\'image\'] or not image.is_file():continue\n    key=record[\'id\']+\' | \'+record[\'role\']+\' | \'+record[\'label\']\n    records[key]=(record,image)\n   except (OSError,ValueError,KeyError):continue\n return records\n\ndef single_image(image):\n if image.ndim!=4 or image.shape[0]!=1 or image.shape[-1]!=3:raise ValueError(\'Use one RGB still at a time.\')\n return Image.fromarray((image[0].detach().cpu().numpy().clip(0,1)*255).round().astype(\'uint8\'))\ndef tensor(image):return torch.from_numpy(np.array(image.convert(\'RGB\'),copy=True).astype(\'float32\')/255).unsqueeze(0)\ndef read_brief(value):\n data=json.loads(value)\n if data.get(\'schema\')!=\'jjp-image-brief-v1\':raise ValueError(\'Connect a JJP Episode Brief to this stage.\')\n return data\n\nclass JJPEpisodeBrief:\n @classmethod\n def INPUT_TYPES(cls):\n  return {\'required\':{\'episode\':(list(PRESETS[\'episodes\']),),\'format\':(list(PRESETS[\'formats\']),),\'scene\':(\'STRING\',{\'multiline\':True,\'default\':\'\'}),\'cast\':([\'Episode default\',\'None\',\'Lead One\',\'Lead Two\',\'Duet\',\'Accuser\',\'Seeker\'],),\'character_prefix\':(\'STRING\',{\'multiline\':True,\'default\':\'\'}),\'title\':(\'STRING\',{\'default\':\'\'}),\'scripture_reference\':(\'STRING\',{\'default\':\'\'}),\'use_episode_reference\':(\'BOOLEAN\',{\'default\':True})},\'optional\':{\'custom_world\':(\'STRING\',{\'multiline\':True,\'default\':\'\'}),\'custom_palette\':(\'STRING\',{\'default\':\'\'}),\'custom_wing\':(\'STRING\',{\'default\':\'\'})}}\n RETURN_TYPES=(\'STRING\',\'STRING\',\'INT\',\'INT\',\'STRING\');RETURN_NAMES=(\'positive\',\'negative\',\'width\',\'height\',\'brief\')\n FUNCTION=\'brief\';CATEGORY=\'JJP/Production\'\n def brief(self,episode,format,scene,cast,character_prefix,title,scripture_reference,use_episode_reference,custom_world=\'\',custom_palette=\'\',custom_wing=\'\'):\n  p=dict(PRESETS[\'episodes\'][episode])\n  if custom_world.strip():p[\'world\']=custom_world.strip()\n  if custom_palette.strip():p[\'palette\']=custom_palette.strip()\n  if custom_wing.strip():p[\'wing\']=custom_wing.strip()\n  role=p[\'cast\'] if cast==\'Episode default\' else cast\n  reference=scripture_reference.strip() or (p[\'reference\'] if use_episode_reference else \'\')\n  prompt=\' \'.join(filter(None,[PRESETS[\'style\'],p[\'world\'],\'Palette: \'+p[\'palette\']+\'.\',PRESETS[\'cast_rule\'],\'Cast for this shot: \'+role+\'.\',character_prefix.strip(),scene.strip()]))\n  w,h=PRESETS[\'formats\'][format]\n  data={\'schema\':\'jjp-image-brief-v1\',\'brand\':PRESETS[\'brand\'],\'episode\':episode,\'wing\':p[\'wing\'],\'palette\':p[\'palette\'],\'cast\':role,\'format\':format,\'title\':title.strip() or (episode if episode!=\'Custom\' else \'\'),\'scripture_reference\':reference,\'reference_status\':\'User must verify the reference against the intended passage; no scripture quotation was generated.\',\'positive\':prompt,\'negative\':PRESETS[\'negative\'],\'character_prefix\':character_prefix,\'generation_size\':[w,h],\'source_document\':\'JJP Launch System, Edition I\',\'rights_notes\':[\'Generation and likeness fidelity require human review.\',\'Commercial-v7 uses Apache-2.0/BSD-3-Clause assets; preserve provenance.json and license notices. Generative identity edits require likeness review.\'],\'review\':[\'Compare human likeness with approved reference; reject drift.\',\'Confirm non-human/faceless antagonist design.\',\'Verify scripture reference and any supplied quotation.\',\'Check title spelling, composition and readable type.\',\'Review likeness dignity and synthetic-content disclosure when publishing.\']}\n  return prompt,PRESETS[\'negative\'],w,h,json.dumps(data,ensure_ascii=False)\n\nclass JJPSaveApprovedCast:\n @classmethod\n def INPUT_TYPES(cls):return {\'required\':{\'image\':(\'IMAGE\',),\'role\':(CAST_ROLES,),\'label\':(\'STRING\',{\'default\':\'Approved reference\'}),\'prompt_prefix\':(\'STRING\',{\'multiline\':True,\'default\':\'\'}),\'approved\':(\'BOOLEAN\',{\'default\':False})}}\n RETURN_TYPES=(\'IMAGE\',\'STRING\');RETURN_NAMES=(\'reference\',\'record\');FUNCTION=\'save\';CATEGORY=\'JJP/Cast\';OUTPUT_NODE=True\n def save(self,image,role,label,prompt_prefix,approved):\n  if not approved:raise ValueError(\'Inspect the reference and set approved=true only when you want to add it to your cast library.\')\n  im=single_image(image);ident=uuid.uuid4().hex[:12];root=output_root()/\'cast\'/slug(role);root.mkdir(parents=True,exist_ok=True)\n  path=root/(slug(label)+\'-\'+ident+\'.png\');im.save(path)\n  record={\'schema\':\'jjp-cast-v1\',\'id\':ident,\'role\':role,\'label\':label.strip() or \'Approved reference\',\'prompt_prefix\':prompt_prefix,\'approved\':True,\'image\':path.name,\'sha256\':hashlib.sha256(path.read_bytes()).hexdigest()}\n  path.with_suffix(\'.cast.json\').write_text(json.dumps(record,indent=2,ensure_ascii=False))\n  return {\'ui\':{\'text\':[f\'Saved approved cast reference: {record["role"]} / {record["label"]}. Save private outputs to preserve your library.\']},\'result\':(image,json.dumps(record,ensure_ascii=False))}\n\nclass JJPLoadApprovedCast:\n @classmethod\n def INPUT_TYPES(cls):return {\'required\':{\'reference\':(list(cast_records()) or [\'Add approved cast reference first\'],)}}\n RETURN_TYPES=(\'IMAGE\',\'STRING\',\'STRING\');RETURN_NAMES=(\'reference_image\',\'character_prefix\',\'cast_record\');FUNCTION=\'load\';CATEGORY=\'JJP/Cast\'\n @classmethod\n def IS_CHANGED(cls,reference):\n  found=cast_records().get(reference)\n  return hashlib.sha256(found[1].read_bytes()).hexdigest() if found else float(\'nan\')\n def load(self,reference):\n  found=cast_records().get(reference)\n  if not found:raise ValueError(\'No matching approved reference. Use JJP - Approve Cast, then refresh the page/node options, or attach your saved private cast output.\')\n  record,path=found\n  if hashlib.sha256(path.read_bytes()).hexdigest()!=record[\'sha256\']:raise ValueError(\'Approved cast file changed; review and register it again.\')\n  with Image.open(path) as im:reference_tensor=tensor(im)\n  return reference_tensor,record[\'prompt_prefix\'],json.dumps(record,ensure_ascii=False)\n\nclass JJPCombineCast:\n @classmethod\n def INPUT_TYPES(cls):return {\'required\':{\'lead_one\':(\'STRING\',{\'forceInput\':True}),\'lead_two\':(\'STRING\',{\'forceInput\':True})}}\n RETURN_TYPES=(\'STRING\',\'STRING\');RETURN_NAMES=(\'character_prefix\',\'cast_record\');FUNCTION=\'combine\';CATEGORY=\'JJP/Cast\'\n def combine(self,lead_one,lead_two):\n  first,second=json.loads(lead_one),json.loads(lead_two)\n  if first.get(\'role\')!=\'Lead One\' or second.get(\'role\')!=\'Lead Two\':raise ValueError(\'Select an approved Lead One reference and an approved Lead Two reference for this duet.\')\n  prefix=\'Lead One: \'+first.get(\'prompt_prefix\',\'\')+\'; Lead Two: \'+second.get(\'prompt_prefix\',\'\')\n  return prefix,json.dumps({\'schema\':\'jjp-cast-group-v1\',\'references\':[first,second]},ensure_ascii=False)\n\ndef font(size):\n for path in [\'/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf\',\'/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf\']:\n  if Path(path).is_file():return ImageFont.truetype(path,size)\n return ImageFont.load_default(size=size)\ndef draw_fit(draw,text,box,fill,start_size):\n if not text:return\n x,y,w,h=box\n for size in range(start_size,11,-1):\n  f=font(size);words=text.split();lines=[];line=\'\'\n  for word in words:\n   trial=(line+\' \'+word).strip()\n   if draw.textlength(trial,font=f)>w and line:lines.append(line);line=word\n   else:line=trial\n  if line:lines.append(line)\n  line_height=size+6\n  if len(lines)*line_height<=h and all(draw.textlength(t,font=f)<=w for t in lines):break\n else:raise ValueError(\'Text is too long for the selected export. Shorten the title or reference.\')\n for i,line in enumerate(lines):draw.text((x,y+i*line_height),line,font=f,fill=fill)\n\nclass JJPArtworkExport:\n @classmethod\n def INPUT_TYPES(cls):\n  return {\'required\':{\'image\':(\'IMAGE\',),\'brief\':(\'STRING\',{\'forceInput\':True}),\'layout\':([\'From brief\',\'Original size\',\'Square cover\',\'YouTube 16:9\',\'Vertical 9:16\'],),\'lettering\':(\'BOOLEAN\',{\'default\':True}),\'long_edge\':(\'INT\',{\'default\':2048,\'min\':256,\'max\':8192,\'step\':64})},\'optional\':{\'cast_record\':(\'STRING\',{\'forceInput\':True})},\'hidden\':{\'prompt\':\'PROMPT\',\'extra_pnginfo\':\'EXTRA_PNGINFO\'}}\n RETURN_TYPES=(\'IMAGE\',\'STRING\');RETURN_NAMES=(\'delivery\',\'saved_files\');FUNCTION=\'export\';CATEGORY=\'JJP/Delivery\';OUTPUT_NODE=True\n def export(self,image,brief,layout,lettering,long_edge,cast_record=None,prompt=None,extra_pnginfo=None):\n  data=read_brief(brief);master=single_image(image);ident=datetime.datetime.now(datetime.timezone.utc).strftime(\'%Y%m%dT%H%M%S\')+\'-\'+uuid.uuid4().hex[:8]\n  root=output_root()/\'episodes\'/slug(data[\'episode\'])/ident;root.mkdir(parents=True,exist_ok=False)\n  meta=PngImagePlugin.PngInfo();meta.add_text(\'jjp_brief\',brief)\n  if prompt is not None:meta.add_text(\'prompt\',json.dumps(prompt))\n  for key,val in (extra_pnginfo or {}).items():meta.add_text(key,json.dumps(val))\n  master.save(root/\'master.png\',pnginfo=meta)\n  # Padding preserves the whole source frame. Interpolation is explicitly reported.\n  if layout==\'From brief\':layout={\'Square cover\':\'Square cover\',\'YouTube thumbnail\':\'YouTube 16:9\',\'Vertical still\':\'Vertical 9:16\'}.get(data[\'format\'],\'Original size\')\n  if layout==\'Original size\':canvas=master.copy()\n  else:\n   ratio={\'Square cover\':(1,1),\'YouTube 16:9\':(16,9),\'Vertical 9:16\':(9,16)}[layout]\n   w=long_edge if ratio[0]>=ratio[1] else round(long_edge*ratio[0]/ratio[1]);h=long_edge if ratio[1]>=ratio[0] else round(long_edge*ratio[1]/ratio[0])\n   canvas=Image.new(\'RGB\',(w,h),\'#101014\');art=ImageOps.contain(master,(w,h),Image.Resampling.LANCZOS);canvas.paste(art,((w-art.width)//2,(h-art.height)//2))\n  if lettering:\n   w,h=canvas.size;draw=ImageDraw.Draw(canvas);band=max(110,round(h*0.23));draw.rectangle((0,h-band,w,h),fill=\'#101014\')\n   margin=max(12,round(w*0.04));base=max(14,round(min(w,h)*0.023))\n   draw_fit(draw,PRESETS[\'brand\'],(margin,h-band+10,w-2*margin,base+12),\'#d9b56b\',base)\n   draw_fit(draw,data[\'title\'],(margin,h-band+base+25,w-2*margin,int(band*0.43)),\'#f6f2e9\',max(20,round(min(w,h)*0.05)))\n   draw_fit(draw,data[\'scripture_reference\'],(margin,h-max(35,base+15),w-2*margin,base+12),\'#d9b56b\',base)\n  canvas.save(root/\'delivery.png\',pnginfo=meta)\n  if cast_record:data[\'cast_reference\']=json.loads(cast_record)\n  data[\'model_profile\']=\'commercial-v7\'\n  provenance=Path(folder_paths.get_output_directory()).parent/\'provenance.json\'\n  if provenance.exists():data[\'model_provenance\']=json.loads(provenance.read_text())\n  data.update({\'run_id\':ident,\'files\':[\'master.png\',\'delivery.png\',\'episode.json\'],\'delivery_layout\':layout,\'delivery_size\':list(canvas.size),\'master_size\':list(master.size),\'delivery_resize\':\'Lanczos interpolation/padding only; use the neural upscaling graph before export for learned detail.\',\'lettering\':lettering,\'approval_status\':\'Needs human review; export does not approve likeness, scripture, model rights or publication.\'})\n  (root/\'episode.json\').write_text(json.dumps(data,indent=2,ensure_ascii=False))\n  summary=\'Saved master.png, delivery.png and episode.json at \'+str(root)\n  return {\'ui\':{\'images\':[{\'filename\':\'delivery.png\',\'subfolder\':str(root.relative_to(Path(folder_paths.get_output_directory()))),\'type\':\'output\'}],\'text\':[summary]},\'result\':(tensor(canvas),summary)}\n\nNODE_CLASS_MAPPINGS={c.__name__:c for c in [JJPEpisodeBrief,JJPSaveApprovedCast,JJPLoadApprovedCast,JJPCombineCast,JJPArtworkExport]}\nNODE_DISPLAY_NAME_MAPPINGS={\'JJPEpisodeBrief\':\'JJP Episode / Shot Brief\',\'JJPSaveApprovedCast\':\'JJP Approve Cast Reference\',\'JJPLoadApprovedCast\':\'JJP Load Approved Cast\',\'JJPCombineCast\':\'JJP Two-Character Cast\',\'JJPArtworkExport\':\'JJP Artwork + Episode Export\'}\n'
JJP_PRESETS_SOURCE = '{\n  "brand": "JesusJuiceProductions",\n  "source": "User-supplied JJP Launch System, Edition I; image-production adaptation",\n  "style": "cinematic live-action realism, grounded physical materials, thoughtful film lighting, dignified performances, no rendered lettering",\n  "negative": "anime, cartoon, manga, illustration, plastic skin, watermark, illegible text, human-faced villain, shame-based imagery, exploitative grief imagery",\n  "episodes": {\n    "Custom": {"wing":"Custom","world":"","palette":"neutral cinematic tones","cast":"None","reference":""},\n    "Verdict of the Lamb": {"wing":"Gothic Gospel","world":"A cosmic courtroom, thunder beyond monumental arches, broken chains, a faceless shadow accuser made of smoke; the human leads stand in mercy and resolve.","palette":"crimson light against charcoal black, restrained ivory highlights","cast":"Duet","reference":"John 19:30"},\n    "Anastasis": {"wing":"Gothic Gospel","world":"The courtroom world opens into an empty-tomb garden at dawn, darkness transforming into resurrection light; no accuser present.","palette":"crimson transitioning to sunrise gold and living green","cast":"Duet","reference":""},\n    "God-Breathed": {"wing":"Doom / War","world":"A twilight battlefield with broken swords, illuminated scripture scrolls and chains opening into light; a lone warrior-scribe faces non-human darkness.","palette":"charcoal, deep crimson, warm scroll light","cast":"Lead One","reference":"2 Timothy 3:16"},\n    "Every Thought Captive": {"wing":"Electronic / Mind","world":"A symbolic war room inside a mind, neural grids and faceless glitch-static entities giving way to a quiet river of light.","palette":"dark indigo, cyan grid light, white peace","cast":"Duet","reference":""},\n    "Present Your Body": {"wing":"Reggae / Garden","world":"A sunset hillside becomes a starlit garden; opened shackles and new growth communicate renewal without spectacle.","palette":"golden hour amber, living green, warm night blue","cast":"Duet","reference":"Romans 12:1"},\n    "Jesus, Thank You For Loving Me": {"wing":"Ballad / Sunrise","world":"An intimate empty-tomb garden at sunrise; one grateful worshipper, soft light, no armies or towers.","palette":"soft sunrise gold, warm ivory, muted garden green","cast":"Lead One","reference":"John 20:16"},\n    "Mirror of Glory": {"wing":"Mystic / Desert","world":"A lone mirror in a golden desert reflects a transformed world of light; the seeker is faceless and cloaked.","palette":"sandstone gold, desert blue, luminous ivory","cast":"Duet","reference":"James 1:23-25; 2 Corinthians 3:18"},\n    "Love Your Enemies": {"wing":"Chamber / Hillside","world":"A Galilean hillside in a sun-shower, rain and light falling across the whole scene; the second lead carries the moment and the first is a distant silent listener.","palette":"rain silver, fresh green, gentle gold","cast":"Lead Two","reference":"Matthew 5:45"}\n  },\n  "formats": {"Episode key art":[1216,832],"Square cover":[1024,1024],"YouTube thumbnail":[1216,832],"Vertical still":[832,1216],"Cast reference":[832,1216]},\n  "cast_rule": "Only approved human leads represent the recurring cast; evil entities remain non-human and faceless. The seeker is faceless and cloaked."\n}\n'
CATALOG = {'profile': 'commercial-v7', 'reviewed': '2026-11-09', 'assets': [{'pack': 'create_edit', 'folder': 'diffusion_models', 'name': 'flux-2-klein-4b.safetensors', 'bytes': 7751105712, 'sha256': 'ec3d4e733a771f61c052fb4856c48b336c55eaf2c65487c2a1faeb9bbda7a343', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/diffusion_models/flux-2-klein-4b.safetensors'}, {'pack': 'create_edit', 'folder': 'text_encoders', 'name': 'qwen_3_4b.safetensors', 'bytes': 8044982048, 'sha256': '6c671498573ac2f7a5501502ccce8d2b08ea6ca2f661c458e708f36b36edfc5a', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/text_encoders/qwen_3_4b.safetensors'}, {'pack': 'create_edit', 'folder': 'vae', 'name': 'flux2-vae.safetensors', 'bytes': 336211292, 'sha256': '868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/vae/flux2-vae.safetensors'}, {'pack': 'finish', 'folder': 'upscale_models', 'name': 'RealESRGAN_x4plus.pth', 'bytes': 67040989, 'sha256': '4fa0d38905f75ac06eb49a7951b426670021be3018265fd191d2125df9d682f1', 'license': 'BSD-3-Clause', 'source': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'}, {'pack': 'finish', 'folder': 'diffusion_models', 'name': 'seedvr2_3b_int8_convrot.safetensors', 'bytes': 3458259704, 'sha256': 'c3dec8bcc5916843a8a858572970597462e1f2dc598d6dfd818f6cd40f53a157', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/diffusion_models/seedvr2_3b_int8_convrot.safetensors'}, {'pack': 'finish', 'folder': 'vae', 'name': 'seedvr2_ema_vae_fp16.safetensors', 'bytes': 501324814, 'sha256': '20678548f420d98d26f11442d3528f8b8c94e57ee046ef93dbb7633da8612ca1', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/vae/seedvr2_ema_vae_fp16.safetensors'}, {'pack': 'finish', 'folder': 'mediapipe', 'name': 'face_landmarker.task', 'bytes': 3758596, 'sha256': '64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff', 'license': 'Apache-2.0', 'source': 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'}, {'pack': 'finish', 'folder': 'loras', 'name': 'bfs_head_v1_flux-klein_4b.safetensors', 'bytes': 369644384, 'sha256': '77aabcd6144612dd4873080b99283dd9a27977712548bee161894f62c91ff09c', 'license': 'MIT', 'source': 'https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/bfs_head_v1_flux-klein_4b.safetensors'}]}
LICENSE_FILES = {'realesrgan-BSD.txt': 'BSD 3-Clause License\n\nCopyright (c) 2021, Xintao Wang\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n1. Redistributions of source code must retain the above copyright notice, this\n   list of conditions and the following disclaimer.\n\n2. Redistributions in binary form must reproduce the above copyright notice,\n   this list of conditions and the following disclaimer in the documentation\n   and/or other materials provided with the distribution.\n\n3. Neither the name of the copyright holder nor the names of its\n   contributors may be used to endorse or promote products derived from\n   this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'mediapipe-model-evidence.txt': 'MediaPipe model license evidence, reviewed 2026-11-09. Bundle SHA256: 64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff\n\nhttps://storage.googleapis.com/mediapipe-assets/Model%20Card%20MediaPipe%20Face%20Mesh%20V2.pdf\nGeng Yan, Google\nIvan Grishchenko, Google\nMODEL DATE\nSeptember 15, 2022\nLICENSED UNDER\nApache License, Version 2.0\n\n_\nBased on Model Cards for Model Repo ing, In Proceedings of FAT* Conference (FAT*2019). ACM, New York, NY, USA\n\nhttps://storage.googleapis.com/mediapipe-assets/MediaPipe%20BlazeFace%20Model%20Card%20(Short%20Range).pdf\n\nDOCUMENTATION\nPaper: https://arxiv.org/abs/2006.10204\n\nLICENSED UNDER\nApache License, Version 2.0\n\nBased on Model Cards for Model Reporting, In Proceedings of FAT* Conference (FAT*2019). ACM, New York, NY, USA\n\n\nhttps://storage.googleapis.com/mediapipe-assets/Model%20Card%20Blendshape%20V2.pdf\nAppendix.\n\nfront-facing camera\n\nLICENSED UNDER\nApache License, Version 2.0\nLe : Input frame with predicted facial landmarks.\nRight: 3D avatar controlled with predicted blendshape\ncoe cients\n', 'flux2-klein4b-APACHE.txt': '                             Apache License\n                       Version 2.0, January 2004\n                    http://www.apache.org/licenses/\n\nTERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n1. Definitions.\n\n  "License" shall mean the terms and conditions for use, reproduction,\n  and distribution as defined by Sections 1 through 9 of this document.\n\n  "Licensor" shall mean the copyright owner or entity authorized by\n  the copyright owner that is granting the License.\n\n  "Legal Entity" shall mean the union of the acting entity and all\n  other entities that control, are controlled by, or are under common\n  control with that entity. For the purposes of this definition,\n  "control" means (i) the power, direct or indirect, to cause the\n  direction or management of such entity, whether by contract or\n  otherwise, or (ii) ownership of fifty percent (50%) or more of the\n  outstanding shares, or (iii) beneficial ownership of such entity.\n\n  "You" (or "Your") shall mean an individual or Legal Entity\n  exercising permissions granted by this License.\n\n  "Source" form shall mean the preferred form for making modifications,\n  including but not limited to software source code, documentation\n  source, and configuration files.\n\n  "Object" form shall mean any form resulting from mechanical\n  transformation or translation of a Source form, including but\n  not limited to compiled object code, generated documentation,\n  and conversions to other media types.\n\n  "Work" shall mean the work of authorship, whether in Source or\n  Object form, made available under the License, as indicated by a\n  copyright notice that is included in or attached to the work\n  (an example is provided in the Appendix below).\n\n  "Derivative Works" shall mean any work, whether in Source or Object\n  form, that is based on (or derived from) the Work and for which the\n  editorial revisions, annotations, elaborations, or other modifications\n  represent, as a whole, an original work of authorship. For the purposes\n  of this License, Derivative Works shall not include works that remain\n  separable from, or merely link (or bind by name) to the interfaces of,\n  the Work and Derivative Works thereof.\n\n  "Contribution" shall mean any work of authorship, including\n  the original version of the Work and any modifications or additions\n  to that Work or Derivative Works thereof, that is intentionally\n  submitted to Licensor for inclusion in the Work by the copyright owner\n  or by an individual or Legal Entity authorized to submit on behalf of\n  the copyright owner. For the purposes of this definition, "submitted"\n  means any form of electronic, verbal, or written communication sent\n  to the Licensor or its representatives, including but not limited to\n  communication on electronic mailing lists, source code control systems,\n  and issue tracking systems that are managed by, or on behalf of, the\n  Licensor for the purpose of discussing and improving the Work, but\n  excluding communication that is conspicuously marked or otherwise\n  designated in writing by the copyright owner as "Not a Contribution."\n\n  "Contributor" shall mean Licensor and any individual or Legal Entity\n  on behalf of whom a Contribution has been received by Licensor and\n  subsequently incorporated within the Work.\n\n2. Grant of Copyright License. Subject to the terms and conditions of\n  this License, each Contributor hereby grants to You a perpetual,\n  worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n  copyright license to reproduce, prepare Derivative Works of,\n  publicly display, publicly perform, sublicense, and distribute the\n  Work and such Derivative Works in Source or Object form.\n\n3. Grant of Patent License. Subject to the terms and conditions of\n  this License, each Contributor hereby grants to You a perpetual,\n  worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n  (except as stated in this section) patent license to make, have made,\n  use, offer to sell, sell, import, and otherwise transfer the Work,\n  where such license applies only to those patent claims licensable\n  by such Contributor that are necessarily infringed by their\n  Contribution(s) alone or by combination of their Contribution(s)\n  with the Work to which such Contribution(s) was submitted. If You\n  institute patent litigation against any entity (including a\n  cross-claim or counterclaim in a lawsuit) alleging that the Work\n  or a Contribution incorporated within the Work constitutes direct\n  or contributory patent infringement, then any patent licenses\n  granted to You under this License for that Work shall terminate\n  as of the date such litigation is filed.\n\n4. Redistribution. You may reproduce and distribute copies of the\n  Work or Derivative Works thereof in any medium, with or without\n  modifications, and in Source or Object form, provided that You\n  meet the following conditions:\n\n  (a) You must give any other recipients of the Work or\n      Derivative Works a copy of this License; and\n\n  (b) You must cause any modified files to carry prominent notices\n      stating that You changed the files; and\n\n  (c) You must retain, in the Source form of any Derivative Works\n      that You distribute, all copyright, patent, trademark, and\n      attribution notices from the Source form of the Work,\n      excluding those notices that do not pertain to any part of\n      the Derivative Works; and\n\n  (d) If the Work includes a "NOTICE" text file as part of its\n      distribution, then any Derivative Works that You distribute must\n      include a readable copy of the attribution notices contained\n      within such NOTICE file, excluding those notices that do not\n      pertain to any part of the Derivative Works, in at least one\n      of the following places: within a NOTICE text file distributed\n      as part of the Derivative Works; within the Source form or\n      documentation, if provided along with the Derivative Works; or,\n      within a display generated by the Derivative Works, if and\n      wherever such third-party notices normally appear. The contents\n      of the NOTICE file are for informational purposes only and\n      do not modify the License. You may add Your own attribution\n      notices within Derivative Works that You distribute, alongside\n      or as an addendum to the NOTICE text from the Work, provided\n      that such additional attribution notices cannot be construed\n      as modifying the License.\n\n  You may add Your own copyright statement to Your modifications and\n  may provide additional or different license terms and conditions\n  for use, reproduction, or distribution of Your modifications, or\n  for any such Derivative Works as a whole, provided Your use,\n  reproduction, and distribution of the Work otherwise complies with\n  the conditions stated in this License.\n\n5. Submission of Contributions. Unless You explicitly state otherwise,\n  any Contribution intentionally submitted for inclusion in the Work\n  by You to the Licensor shall be under the terms and conditions of\n  this License, without any additional terms or conditions.\n  Notwithstanding the above, nothing herein shall supersede or modify\n  the terms of any separate license agreement you may have executed\n  with Licensor regarding such Contributions.\n\n6. Trademarks. This License does not grant permission to use the trade\n  names, trademarks, service marks, or product names of the Licensor,\n  except as required for reasonable and customary use in describing the\n  origin of the Work and reproducing the content of the NOTICE file.\n\n7. Disclaimer of Warranty. Unless required by applicable law or\n  agreed to in writing, Licensor provides the Work (and each\n  Contributor provides its Contributions) on an "AS IS" BASIS,\n  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n  implied, including, without limitation, any warranties or conditions\n  of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n  PARTICULAR PURPOSE. You are solely responsible for determining the\n  appropriateness of using or redistributing the Work and assume any\n  risks associated with Your exercise of permissions under this License.\n\n8. Limitation of Liability. In no event and under no legal theory,\n  whether in tort (including negligence), contract, or otherwise,\n  unless required by applicable law (such as deliberate and grossly\n  negligent acts) or agreed to in writing, shall any Contributor be\n  liable to You for damages, including any direct, indirect, special,\n  incidental, or consequential damages of any character arising as a\n  result of this License or out of the use or inability to use the\n  Work (including but not limited to damages for loss of goodwill,\n  work stoppage, computer failure or malfunction, or any and all\n  other commercial damages or losses), even if such Contributor\n  has been advised of the possibility of such damages.\n\n9. Accepting Warranty or Additional Liability. While redistributing\n  the Work or Derivative Works thereof, You may choose to offer,\n  and charge a fee for, acceptance of support, warranty, indemnity,\n  or other liability obligations and/or rights consistent with this\n  License. However, in accepting such obligations, You may act only\n  on Your own behalf and on Your sole responsibility, not on behalf\n  of any other Contributor, and only if You agree to indemnify,\n  defend, and hold each Contributor harmless for any liability\n  incurred by, or claims asserted against, such Contributor by reason\n  of your accepting any such warranty or additional liability.\n\nEND OF TERMS AND CONDITIONS\n', 'bfs-model-card-MIT.md': 'Original model card snapshot, declared MIT. Source: https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/blob/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/README.md\n\n---\nlicense: mit\nbase_model:\n- Qwen/Qwen-Image-2.1\n- Qwen/Qwen-Image-Edit-2511\n- black-forest-labs/FLUX.2-klein-9B\n- black-forest-labs/FLUX.2-klein-4B\n- black-forest-labs/FLUX.2-klein-base-4B\n- black-forest-labs/FLUX.2-klein-base-9B\n- black-forest-labs/FLUX.2-klein-9b-fp8\n- black-forest-labs/FLUX.2-klein-4b-fp8\n- krea/Krea-2-Raw\npipeline_tag: image-to-image\ntags:\n- lora\n- qwen-image\n- qwen-image-2.1\n- qwen-image-edit\n- face-swap\n- head-swap\n- body-swap\n- selective-body-swap\n- image-editing\n- comfyui\n- bfs\nlanguage:\n- en\nlibrary_name: diffusers\n---\n\n# BFS: Best Face Swap\n\nBFS is a collection of LoRAs for face, head, and body replacement across Qwen Image 2.1, Qwen\nImage Edit 2509/2511, Flux 2 Klein, Krea 2, and experimental LTX-2 workflows. Each release has\nits own base-model, input-order, prompt, and workflow contract. Use the guide for the exact\nversion you are loading.\n\n## Guides\n\n- [BFS Head V1 / V1.1 for Qwen Image 2.1](docs/qwen-image-2.1.md)\n- [BFS Body Swap V1.0 for Qwen Image 2.1](docs/qwen-image-2.1-body.md)\n- [BFS Body Swap Selective (Magenta) V1.0 for Qwen Image 2.1](docs/qwen-image-2.1-body-selective.md):\n  replaces only the person marked in magenta, for group photos and busy scenes\n- [Qwen Image Edit 2509/2511](docs/qwen-image-edit.md)\n- [Flux 2 Klein](docs/flux-2-klein.md)\n- [Krea 2](docs/krea-2.md)\n- [Experimental LTX-2 weights](docs/ltx-2.md)\n\n## Downloads at a glance\n\n| Family | Weights | Workflow |\n|---|---|---|\n| Qwen Image 2.1 (head) | [`bfs_head_v1.1_qwen_2.1.safetensors`](bfs_head_v1.1_qwen_2.1.safetensors) (recommended), [`bfs_head_v1.1_alternative_qwen_2.1.safetensors`](bfs_head_v1.1_alternative_qwen_2.1.safetensors), [`bfs_head_v1_qwen_2.1.safetensors`](bfs_head_v1_qwen_2.1.safetensors) | [`Head Swap V1 Qwen 2.1 Workflow.json`](workflows/Head%20Swap%20V1%20Qwen%202.1%20Workflow.json) |\n| Qwen Image 2.1 (body) | [`bfs_body_swap_v1.0_qwen_2.1.safetensors`](bfs_body_swap_v1.0_qwen_2.1.safetensors) | [`Body Swap Qwen Image 2.1 - V1.json`](workflows/Body%20Swap%20Qwen%20Image%202.1%20-%20V1.json), see the [body swap guide](docs/qwen-image-2.1-body.md) |\n| Qwen Image 2.1 (selective body, magenta mark) | [`bfs_body_swap_selective_magenta_v1.0_qwen_2.1.safetensors`](bfs_body_swap_selective_magenta_v1.0_qwen_2.1.safetensors) | [`Body Swap Selective Qwen Image 2.1 - V1.0.json`](workflows/Body%20Swap%20Selective%20Qwen%20Image%202.1%20-%20V1.0.json), see the [selective guide](docs/qwen-image-2.1-body-selective.md) |\n| Qwen Image Edit 2509 | [`bfs_face_v1_qwen_image_edit_2509.safetensors`](bfs_face_v1_qwen_image_edit_2509.safetensors), [`bfs_head_v1_qwen_image_edit_2509.safetensors`](bfs_head_v1_qwen_image_edit_2509.safetensors), [`bfs_head_v2_qwen_image_edit_2509.safetensors`](bfs_head_v2_qwen_image_edit_2509.safetensors), [`bfs_head_v3_qwen_image_edit_2509.safetensors`](bfs_head_v3_qwen_image_edit_2509.safetensors), [`bfs_head_v4_qwen_image_edit_2509.safetensors`](bfs_head_v4_qwen_image_edit_2509.safetensors) | See the [Qwen Edit guide](docs/qwen-image-edit.md) |\n| Qwen Image Edit 2511 | [`bfs_head_v5_2511_original.safetensors`](bfs_head_v5_2511_original.safetensors), [`bfs_head_v5_2511_merged_version_rank_16_fp16.safetensors`](bfs_head_v5_2511_merged_version_rank_16_fp16.safetensors), [`bfs_head_v5_2511_merged_version_rank_32_fp32.safetensors`](bfs_head_v5_2511_merged_version_rank_32_fp32.safetensors) | See the [Qwen Edit guide](docs/qwen-image-edit.md) |\n| Flux 2 Klein | [`bfs_head_v1_flux-klein_4b.safetensors`](bfs_head_v1_flux-klein_4b.safetensors), [`bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors`](bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors), [`bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors`](bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors), [`bfs_head_v1.1_optional_flux-klein_4b.safetensors`](bfs_head_v1.1_optional_flux-klein_4b.safetensors) | See the [Flux guide](docs/flux-2-klein.md) |\n| Krea 2 | [`bfs_head_swap_v1.1_krea2.safetensors`](bfs_head_swap_v1.1_krea2.safetensors), [`bfs_body_swap_v1_krea2.safetensors`](bfs_body_swap_v1_krea2.safetensors) | See the [Krea 2 guide](docs/krea-2.md) |\n| Experimental LTX-2 | [`bfs_head_swap_v1_ltx2_ic_lora_5000.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_5000.safetensors), [`bfs_head_swap_v1_ltx2_ic_lora_6000.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_6000.safetensors), [`bfs_head_swap_v1_ltx2_ic_lora_12000_train_3.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_12000_train_3.safetensors) | See the [LTX-2 guide](docs/ltx-2.md) |\n\n## General rules\n\n- Use the LoRA with its matching base model whenever possible.\n- Follow the input order in the version guide; several BFS head releases intentionally use body\n  first and reference face second.\n- Start at LoRA strength `1.0`, then adjust for the base model, resolution, and reference image.\n- The examples and workflows in this repository are demonstrations, not a guarantee of identical\n  results on every image.\n- Do not use or share results involving public figures or people who have not given consent.\n\n## Examples\n\nThe existing examples are grouped by workflow family in the individual guides. Qwen Image 2.1\nexamples are `images/bfs_qwen_2.1_1.png` through `images/bfs_qwen_2.1_8.png`.\n\n## Support\n\nFeel free to experiment and share your results responsibly. For workflow issues, first update\nComfyUI and verify that the base model, text encoder, VAE, workflow, and LoRA all belong to the\nsame model family.\n', 'seedvr2-APACHE.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "{}"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2025 seed\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'qwen3-4b-APACHE.txt': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2024 Alibaba Cloud\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.', 'bfs-klein-guide.md': 'Source: https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/blob/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/docs/flux-2-klein.md\n\n# Flux 2 Klein Head-Swap LoRAs\n\nThese BFS weights target Flux 2 Klein 4B/9B variants. They use the body-first input contract:\n\n- **Image 1:** target body/base image\n- **Image 2:** reference face/head\n\nDo not reverse the images for these releases.\n\n## Downloads\n\n- [`bfs_head_v1_flux-klein_4b.safetensors`](../bfs_head_v1_flux-klein_4b.safetensors)\n- [`bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors`](../bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors)\n- [`bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors`](../bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors)\n- [`bfs_head_v1.1_optional_flux-klein_4b.safetensors`](../bfs_head_v1.1_optional_flux-klein_4b.safetensors)\n\n## Prompt\n\n```text\nhead_swap: start with Picture 1 as the base image, keeping its lighting, environment, and background. remove the head from Picture 1 completely and replace it with the head from Picture 2, strictly preserving the hair, eye color, and nose structure of Picture 2. copy the direction of the eye, head rotation, and micro-expressions from Picture 1. Keep the body, clothing, pose, framing, and background intact.\n```\n\nYou can add a short expression or pose instruction when it is visible in the target image. Keep\nthe edit focused on the head and avoid requesting an unrelated scene change in the same prompt.\n\n## Workflow\n\n[`workflows/Head Swap V1 Flux 2 Klein 4b_9b (base_distill).json`](../workflows/Head%20Swap%20V1%20Flux%202%20Klein%204b_9b%20(base_distill).json)\n\nStart around LoRA strength `1.0`; adjust after checking identity preservation, hair boundaries,\nexpression transfer, and skin blending.\n\n[Back to the repository index](../README.md)\n', 'mediapipe-source-APACHE.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n\n===========================================================================\nFor files under tasks/cc/text/language_detector/custom_ops/utils/utf/\n===========================================================================\n/*\n * The authors of this software are Rob Pike and Ken Thompson.\n *              Copyright (c) 2002 by Lucent Technologies.\n * Permission to use, copy, modify, and distribute this software for any\n * purpose without fee is hereby granted, provided that this entire notice\n * is included in all copies of any software which is or includes a copy\n * or modification of this software and in all copies of the supporting\n * documentation for such software.\n * THIS SOFTWARE IS BEING PROVIDED "AS IS", WITHOUT ANY EXPRESS OR IMPLIED\n * WARRANTY.  IN PARTICULAR, NEITHER THE AUTHORS NOR LUCENT TECHNOLOGIES MAKE ANY\n * REPRESENTATION OR WARRANTY OF ANY KIND CONCERNING THE MERCHANTABILITY\n * OF THIS SOFTWARE OR ITS FITNESS FOR ANY PARTICULAR PURPOSE.\n */\n'}
WORKFLOW_PAYLOAD = {'Create': {'last_node_id': 14, 'last_link_id': 19, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [4, 6]}, {'name': 'height', 'type': 'INT', 'links': [5, 7]}, {'name': 'brief', 'type': 'STRING', 'links': [19]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', 'A cinematic music video still.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 8, 'type': 'Flux2Scheduler', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 6}, {'name': 'height', 'type': 'INT', 'link': 7}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [14]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 9, 'type': 'RandomNoise', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [11]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 10, 'type': 'KSamplerSelect', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [13]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 11, 'type': 'CFGGuider', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [12]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 12, 'type': 'SamplerCustomAdvanced', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 11}, {'name': 'guider', 'type': 'GUIDER', 'link': 12}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 13}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 14}, {'name': 'latent_image', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [16]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 13, 'type': 'VAEDecodeTiled', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 14, 'type': 'JJPArtworkExport', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}, {'name': 'brief', 'type': 'STRING', 'link': 19}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 1, 2, 7, 0, 'INT'], [5, 1, 3, 7, 1, 'INT'], [6, 1, 2, 8, 0, 'INT'], [7, 1, 3, 8, 1, 'INT'], [8, 2, 0, 11, 0, 'MODEL'], [9, 5, 0, 11, 1, 'CONDITIONING'], [10, 6, 0, 11, 2, 'CONDITIONING'], [11, 9, 0, 12, 0, 'NOISE'], [12, 11, 0, 12, 1, 'GUIDER'], [13, 10, 0, 12, 2, 'SAMPLER'], [14, 8, 0, 12, 3, 'SIGMAS'], [15, 7, 0, 12, 4, 'LATENT'], [16, 12, 0, 13, 0, 'LATENT'], [17, 4, 0, 13, 1, 'VAE'], [18, 13, 0, 14, 0, 'IMAGE'], [19, 1, 4, 14, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Reference Edit': {'last_node_id': 19, 'last_link_id': 26, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [11, 13]}, {'name': 'height', 'type': 'INT', 'links': [12, 14]}, {'name': 'brief', 'type': 'STRING', 'links': [26]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', 'Use image 1 as the scene and composition. Edit image 1 as follows: describe the desired change here.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [15]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [6, 24]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 7]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [9]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 1 — clean target / scene'}, {'id': 8, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 4}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [5]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 9, 'type': 'VAEEncode', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 5}, {'name': 'vae', 'type': 'VAE', 'link': 6}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [8, 10]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 10, 'type': 'ReferenceLatent', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 7}, {'name': 'latent', 'type': 'LATENT', 'link': 8}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [16]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 11, 'type': 'ReferenceLatent', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 9}, {'name': 'latent', 'type': 'LATENT', 'link': 10}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [17]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 12, 'type': 'EmptyFlux2LatentImage', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 11}, {'name': 'height', 'type': 'INT', 'link': 12}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [22]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 13, 'type': 'Flux2Scheduler', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 13}, {'name': 'height', 'type': 'INT', 'link': 14}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [21]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 14, 'type': 'RandomNoise', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [18]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 15, 'type': 'KSamplerSelect', 'pos': [1480, 660], 'size': [330, 280], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [20]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 16, 'type': 'CFGGuider', 'pos': [1850, 0], 'size': [330, 280], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 15}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 16}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 17}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [19]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 17, 'type': 'SamplerCustomAdvanced', 'pos': [1850, 330], 'size': [330, 280], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 18}, {'name': 'guider', 'type': 'GUIDER', 'link': 19}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 20}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 21}, {'name': 'latent_image', 'type': 'LATENT', 'link': 22}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [23]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 18, 'type': 'VAEDecodeTiled', 'pos': [1850, 660], 'size': [330, 280], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 23}, {'name': 'vae', 'type': 'VAE', 'link': 24}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [25]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 19, 'type': 'JJPArtworkExport', 'pos': [2220, 0], 'size': [330, 280], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 25}, {'name': 'brief', 'type': 'STRING', 'link': 26}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 7, 0, 8, 0, 'IMAGE'], [5, 8, 0, 9, 0, 'IMAGE'], [6, 4, 0, 9, 1, 'VAE'], [7, 5, 0, 10, 0, 'CONDITIONING'], [8, 9, 0, 10, 1, 'LATENT'], [9, 6, 0, 11, 0, 'CONDITIONING'], [10, 9, 0, 11, 1, 'LATENT'], [11, 1, 2, 12, 0, 'INT'], [12, 1, 3, 12, 1, 'INT'], [13, 1, 2, 13, 0, 'INT'], [14, 1, 3, 13, 1, 'INT'], [15, 2, 0, 16, 0, 'MODEL'], [16, 10, 0, 16, 1, 'CONDITIONING'], [17, 11, 0, 16, 2, 'CONDITIONING'], [18, 14, 0, 17, 0, 'NOISE'], [19, 16, 0, 17, 1, 'GUIDER'], [20, 15, 0, 17, 2, 'SAMPLER'], [21, 13, 0, 17, 3, 'SIGMAS'], [22, 12, 0, 17, 4, 'LATENT'], [23, 17, 0, 18, 0, 'LATENT'], [24, 4, 0, 18, 1, 'VAE'], [25, 18, 0, 19, 0, 'IMAGE'], [26, 1, 4, 19, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Masked Edit': {'last_node_id': 20, 'last_link_id': 29, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [11, 13]}, {'name': 'height', 'type': 'INT', 'links': [12, 14]}, {'name': 'brief', 'type': 'STRING', 'links': [29]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', 'Use image 1 as the scene and composition. Edit image 1 as follows: describe the desired change here.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [15]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [6, 24]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 7]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [9]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4, 25]}, {'name': 'MASK', 'type': 'MASK', 'links': [27]}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 1 — clean target / scene'}, {'id': 8, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 4}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [5]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 9, 'type': 'VAEEncode', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 5}, {'name': 'vae', 'type': 'VAE', 'link': 6}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [8, 10]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 10, 'type': 'ReferenceLatent', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 7}, {'name': 'latent', 'type': 'LATENT', 'link': 8}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [16]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 11, 'type': 'ReferenceLatent', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 9}, {'name': 'latent', 'type': 'LATENT', 'link': 10}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [17]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 12, 'type': 'EmptyFlux2LatentImage', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 11}, {'name': 'height', 'type': 'INT', 'link': 12}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [22]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 13, 'type': 'Flux2Scheduler', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 13}, {'name': 'height', 'type': 'INT', 'link': 14}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [21]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 14, 'type': 'RandomNoise', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [18]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 15, 'type': 'KSamplerSelect', 'pos': [1480, 660], 'size': [330, 280], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [20]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 16, 'type': 'CFGGuider', 'pos': [1850, 0], 'size': [330, 280], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 15}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 16}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 17}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [19]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 17, 'type': 'SamplerCustomAdvanced', 'pos': [1850, 330], 'size': [330, 280], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 18}, {'name': 'guider', 'type': 'GUIDER', 'link': 19}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 20}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 21}, {'name': 'latent_image', 'type': 'LATENT', 'link': 22}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [23]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 18, 'type': 'VAEDecodeTiled', 'pos': [1850, 660], 'size': [330, 280], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 23}, {'name': 'vae', 'type': 'VAE', 'link': 24}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [26]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 19, 'type': 'CommercialMaskedEdit', 'pos': [2220, 0], 'size': [330, 280], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [{'name': 'original', 'type': 'IMAGE', 'link': 25}, {'name': 'edited', 'type': 'IMAGE', 'link': 26}, {'name': 'mask', 'type': 'MASK', 'link': 27}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [28]}], 'properties': {'Node name for S&R': 'CommercialMaskedEdit'}, 'widgets_values': [12], 'title': 'CommercialMaskedEdit'}, {'id': 20, 'type': 'JJPArtworkExport', 'pos': [2220, 330], 'size': [330, 280], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 28}, {'name': 'brief', 'type': 'STRING', 'link': 29}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 7, 0, 8, 0, 'IMAGE'], [5, 8, 0, 9, 0, 'IMAGE'], [6, 4, 0, 9, 1, 'VAE'], [7, 5, 0, 10, 0, 'CONDITIONING'], [8, 9, 0, 10, 1, 'LATENT'], [9, 6, 0, 11, 0, 'CONDITIONING'], [10, 9, 0, 11, 1, 'LATENT'], [11, 1, 2, 12, 0, 'INT'], [12, 1, 3, 12, 1, 'INT'], [13, 1, 2, 13, 0, 'INT'], [14, 1, 3, 13, 1, 'INT'], [15, 2, 0, 16, 0, 'MODEL'], [16, 10, 0, 16, 1, 'CONDITIONING'], [17, 11, 0, 16, 2, 'CONDITIONING'], [18, 14, 0, 17, 0, 'NOISE'], [19, 16, 0, 17, 1, 'GUIDER'], [20, 15, 0, 17, 2, 'SAMPLER'], [21, 13, 0, 17, 3, 'SIGMAS'], [22, 12, 0, 17, 4, 'LATENT'], [23, 17, 0, 18, 0, 'LATENT'], [24, 4, 0, 18, 1, 'VAE'], [25, 7, 0, 19, 0, 'IMAGE'], [26, 18, 0, 19, 1, 'IMAGE'], [27, 7, 1, 19, 2, 'MASK'], [28, 19, 0, 20, 0, 'IMAGE'], [29, 1, 4, 20, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Identity Edit': {'last_node_id': 24, 'last_link_id': 33, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [18, 20]}, {'name': 'height', 'type': 'INT', 'links': [19, 21]}, {'name': 'brief', 'type': 'STRING', 'links': [33]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', 'Use image 1 as the scene and composition. Replace only the selected person’s identity with the person from image 2. Preserve pose, clothing, lighting and the rest of the scene.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [22]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [6, 13, 31]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 7]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [9]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 1 — clean target / scene'}, {'id': 8, 'type': 'LoadImage', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 2 — identity reference'}, {'id': 9, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 4}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [5]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 10, 'type': 'VAEEncode', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 5}, {'name': 'vae', 'type': 'VAE', 'link': 6}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [8, 10]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 11, 'type': 'ReferenceLatent', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 7}, {'name': 'latent', 'type': 'LATENT', 'link': 8}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [14]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 12, 'type': 'ReferenceLatent', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 9}, {'name': 'latent', 'type': 'LATENT', 'link': 10}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [16]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 13, 'type': 'ImageScaleToTotalPixels', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 11}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [12]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 14, 'type': 'VAEEncode', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 12}, {'name': 'vae', 'type': 'VAE', 'link': 13}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15, 17]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 15, 'type': 'ReferenceLatent', 'pos': [1480, 660], 'size': [330, 280], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 14}, {'name': 'latent', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [23]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 16, 'type': 'ReferenceLatent', 'pos': [1850, 0], 'size': [330, 280], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 16}, {'name': 'latent', 'type': 'LATENT', 'link': 17}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [24]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 17, 'type': 'EmptyFlux2LatentImage', 'pos': [1850, 330], 'size': [330, 280], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 18}, {'name': 'height', 'type': 'INT', 'link': 19}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [29]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 18, 'type': 'Flux2Scheduler', 'pos': [1850, 660], 'size': [330, 280], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 20}, {'name': 'height', 'type': 'INT', 'link': 21}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [28]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 19, 'type': 'RandomNoise', 'pos': [2220, 0], 'size': [330, 280], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [25]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 20, 'type': 'KSamplerSelect', 'pos': [2220, 330], 'size': [330, 280], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [27]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 21, 'type': 'CFGGuider', 'pos': [2220, 660], 'size': [330, 280], 'flags': {}, 'order': 20, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 22}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 23}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 24}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [26]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 22, 'type': 'SamplerCustomAdvanced', 'pos': [2590, 0], 'size': [330, 280], 'flags': {}, 'order': 21, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 25}, {'name': 'guider', 'type': 'GUIDER', 'link': 26}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 27}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 28}, {'name': 'latent_image', 'type': 'LATENT', 'link': 29}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [30]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 23, 'type': 'VAEDecodeTiled', 'pos': [2590, 330], 'size': [330, 280], 'flags': {}, 'order': 22, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 30}, {'name': 'vae', 'type': 'VAE', 'link': 31}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [32]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 24, 'type': 'JJPArtworkExport', 'pos': [2590, 660], 'size': [330, 280], 'flags': {}, 'order': 23, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 32}, {'name': 'brief', 'type': 'STRING', 'link': 33}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 7, 0, 9, 0, 'IMAGE'], [5, 9, 0, 10, 0, 'IMAGE'], [6, 4, 0, 10, 1, 'VAE'], [7, 5, 0, 11, 0, 'CONDITIONING'], [8, 10, 0, 11, 1, 'LATENT'], [9, 6, 0, 12, 0, 'CONDITIONING'], [10, 10, 0, 12, 1, 'LATENT'], [11, 8, 0, 13, 0, 'IMAGE'], [12, 13, 0, 14, 0, 'IMAGE'], [13, 4, 0, 14, 1, 'VAE'], [14, 11, 0, 15, 0, 'CONDITIONING'], [15, 14, 0, 15, 1, 'LATENT'], [16, 12, 0, 16, 0, 'CONDITIONING'], [17, 14, 0, 16, 1, 'LATENT'], [18, 1, 2, 17, 0, 'INT'], [19, 1, 3, 17, 1, 'INT'], [20, 1, 2, 18, 0, 'INT'], [21, 1, 3, 18, 1, 'INT'], [22, 2, 0, 21, 0, 'MODEL'], [23, 15, 0, 21, 1, 'CONDITIONING'], [24, 16, 0, 21, 2, 'CONDITIONING'], [25, 19, 0, 22, 0, 'NOISE'], [26, 21, 0, 22, 1, 'GUIDER'], [27, 20, 0, 22, 2, 'SAMPLER'], [28, 18, 0, 22, 3, 'SIGMAS'], [29, 17, 0, 22, 4, 'LATENT'], [30, 22, 0, 23, 0, 'LATENT'], [31, 4, 0, 23, 1, 'VAE'], [32, 23, 0, 24, 0, 'IMAGE'], [33, 1, 4, 24, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Duet Identity Edit': {'last_node_id': 29, 'last_link_id': 40, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [25, 27]}, {'name': 'height', 'type': 'INT', 'links': [26, 28]}, {'name': 'brief', 'type': 'STRING', 'links': [40]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', 'Use image 1 as the scene and composition. Use image 2 for Lead One and image 3 for Lead Two; keep their identities distinct.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [29]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [6, 13, 20, 38]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 7]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [9]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 1 — clean target / scene'}, {'id': 8, 'type': 'LoadImage', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 2 — identity reference'}, {'id': 9, 'type': 'LoadImage', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 3 — second identity'}, {'id': 10, 'type': 'ImageScaleToTotalPixels', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 4}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [5]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 11, 'type': 'VAEEncode', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 5}, {'name': 'vae', 'type': 'VAE', 'link': 6}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [8, 10]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 12, 'type': 'ReferenceLatent', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 7}, {'name': 'latent', 'type': 'LATENT', 'link': 8}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [14]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 13, 'type': 'ReferenceLatent', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 9}, {'name': 'latent', 'type': 'LATENT', 'link': 10}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [16]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 14, 'type': 'ImageScaleToTotalPixels', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 11}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [12]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 15, 'type': 'VAEEncode', 'pos': [1480, 660], 'size': [330, 280], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 12}, {'name': 'vae', 'type': 'VAE', 'link': 13}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15, 17]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 16, 'type': 'ReferenceLatent', 'pos': [1850, 0], 'size': [330, 280], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 14}, {'name': 'latent', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [21]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 17, 'type': 'ReferenceLatent', 'pos': [1850, 330], 'size': [330, 280], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 16}, {'name': 'latent', 'type': 'LATENT', 'link': 17}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [23]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 18, 'type': 'ImageScaleToTotalPixels', 'pos': [1850, 660], 'size': [330, 280], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [19]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 19, 'type': 'VAEEncode', 'pos': [2220, 0], 'size': [330, 280], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 19}, {'name': 'vae', 'type': 'VAE', 'link': 20}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [22, 24]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 20, 'type': 'ReferenceLatent', 'pos': [2220, 330], 'size': [330, 280], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 21}, {'name': 'latent', 'type': 'LATENT', 'link': 22}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [30]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 21, 'type': 'ReferenceLatent', 'pos': [2220, 660], 'size': [330, 280], 'flags': {}, 'order': 20, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 23}, {'name': 'latent', 'type': 'LATENT', 'link': 24}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [31]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 22, 'type': 'EmptyFlux2LatentImage', 'pos': [2590, 0], 'size': [330, 280], 'flags': {}, 'order': 21, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 25}, {'name': 'height', 'type': 'INT', 'link': 26}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [36]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 23, 'type': 'Flux2Scheduler', 'pos': [2590, 330], 'size': [330, 280], 'flags': {}, 'order': 22, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 27}, {'name': 'height', 'type': 'INT', 'link': 28}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [35]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 24, 'type': 'RandomNoise', 'pos': [2590, 660], 'size': [330, 280], 'flags': {}, 'order': 23, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [32]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 25, 'type': 'KSamplerSelect', 'pos': [2960, 0], 'size': [330, 280], 'flags': {}, 'order': 24, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [34]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 26, 'type': 'CFGGuider', 'pos': [2960, 330], 'size': [330, 280], 'flags': {}, 'order': 25, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 29}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 30}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 31}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [33]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 27, 'type': 'SamplerCustomAdvanced', 'pos': [2960, 660], 'size': [330, 280], 'flags': {}, 'order': 26, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 32}, {'name': 'guider', 'type': 'GUIDER', 'link': 33}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 34}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 35}, {'name': 'latent_image', 'type': 'LATENT', 'link': 36}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [37]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 28, 'type': 'VAEDecodeTiled', 'pos': [3330, 0], 'size': [330, 280], 'flags': {}, 'order': 27, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 37}, {'name': 'vae', 'type': 'VAE', 'link': 38}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [39]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 29, 'type': 'JJPArtworkExport', 'pos': [3330, 330], 'size': [330, 280], 'flags': {}, 'order': 28, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 39}, {'name': 'brief', 'type': 'STRING', 'link': 40}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 7, 0, 10, 0, 'IMAGE'], [5, 10, 0, 11, 0, 'IMAGE'], [6, 4, 0, 11, 1, 'VAE'], [7, 5, 0, 12, 0, 'CONDITIONING'], [8, 11, 0, 12, 1, 'LATENT'], [9, 6, 0, 13, 0, 'CONDITIONING'], [10, 11, 0, 13, 1, 'LATENT'], [11, 8, 0, 14, 0, 'IMAGE'], [12, 14, 0, 15, 0, 'IMAGE'], [13, 4, 0, 15, 1, 'VAE'], [14, 12, 0, 16, 0, 'CONDITIONING'], [15, 15, 0, 16, 1, 'LATENT'], [16, 13, 0, 17, 0, 'CONDITIONING'], [17, 15, 0, 17, 1, 'LATENT'], [18, 9, 0, 18, 0, 'IMAGE'], [19, 18, 0, 19, 0, 'IMAGE'], [20, 4, 0, 19, 1, 'VAE'], [21, 16, 0, 20, 0, 'CONDITIONING'], [22, 19, 0, 20, 1, 'LATENT'], [23, 17, 0, 21, 0, 'CONDITIONING'], [24, 19, 0, 21, 1, 'LATENT'], [25, 1, 2, 22, 0, 'INT'], [26, 1, 3, 22, 1, 'INT'], [27, 1, 2, 23, 0, 'INT'], [28, 1, 3, 23, 1, 'INT'], [29, 2, 0, 26, 0, 'MODEL'], [30, 20, 0, 26, 1, 'CONDITIONING'], [31, 21, 0, 26, 2, 'CONDITIONING'], [32, 24, 0, 27, 0, 'NOISE'], [33, 26, 0, 27, 1, 'GUIDER'], [34, 25, 0, 27, 2, 'SAMPLER'], [35, 23, 0, 27, 3, 'SIGMAS'], [36, 22, 0, 27, 4, 'LATENT'], [37, 27, 0, 28, 0, 'LATENT'], [38, 4, 0, 28, 1, 'VAE'], [39, 28, 0, 29, 0, 'IMAGE'], [40, 1, 4, 29, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'BFS Head Swap 4B': {'last_node_id': 25, 'last_link_id': 34, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [18, 20]}, {'name': 'height', 'type': 'INT', 'links': [19, 21]}, {'name': 'brief', 'type': 'STRING', 'links': [33]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', 'head_swap: Use Picture 1 as the base scene. Replace its head with the head from Picture 2, preserving the reference hair, eyes, nose and identity. Match the target head rotation, gaze and expression. Keep body, clothing, lighting, framing and background intact.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [22]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [6, 13, 31]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 7]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [9]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 1 — clean target / scene'}, {'id': 8, 'type': 'LoadImage', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 2 — identity reference'}, {'id': 9, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 4}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [5]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 10, 'type': 'VAEEncode', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 5}, {'name': 'vae', 'type': 'VAE', 'link': 6}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [8, 10]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 11, 'type': 'ReferenceLatent', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 7}, {'name': 'latent', 'type': 'LATENT', 'link': 8}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [14]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 12, 'type': 'ReferenceLatent', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 9}, {'name': 'latent', 'type': 'LATENT', 'link': 10}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [16]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 13, 'type': 'ImageScaleToTotalPixels', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 11}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [12]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 14, 'type': 'VAEEncode', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 12}, {'name': 'vae', 'type': 'VAE', 'link': 13}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15, 17]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 15, 'type': 'ReferenceLatent', 'pos': [1480, 660], 'size': [330, 280], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 14}, {'name': 'latent', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [23]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 16, 'type': 'ReferenceLatent', 'pos': [1850, 0], 'size': [330, 280], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 16}, {'name': 'latent', 'type': 'LATENT', 'link': 17}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [24]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 17, 'type': 'EmptyFlux2LatentImage', 'pos': [1850, 330], 'size': [330, 280], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 18}, {'name': 'height', 'type': 'INT', 'link': 19}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [29]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 18, 'type': 'Flux2Scheduler', 'pos': [1850, 660], 'size': [330, 280], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 20}, {'name': 'height', 'type': 'INT', 'link': 21}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [28]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 19, 'type': 'RandomNoise', 'pos': [2220, 0], 'size': [330, 280], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [25]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 20, 'type': 'KSamplerSelect', 'pos': [2220, 330], 'size': [330, 280], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [27]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 21, 'type': 'CFGGuider', 'pos': [2220, 660], 'size': [330, 280], 'flags': {}, 'order': 20, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 34}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 23}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 24}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [26]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 22, 'type': 'SamplerCustomAdvanced', 'pos': [2590, 0], 'size': [330, 280], 'flags': {}, 'order': 21, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 25}, {'name': 'guider', 'type': 'GUIDER', 'link': 26}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 27}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 28}, {'name': 'latent_image', 'type': 'LATENT', 'link': 29}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [30]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 23, 'type': 'VAEDecodeTiled', 'pos': [2590, 330], 'size': [330, 280], 'flags': {}, 'order': 22, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 30}, {'name': 'vae', 'type': 'VAE', 'link': 31}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [32]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 24, 'type': 'JJPArtworkExport', 'pos': [2590, 660], 'size': [330, 280], 'flags': {}, 'order': 23, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 32}, {'name': 'brief', 'type': 'STRING', 'link': 33}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}, {'id': 25, 'type': 'LoraLoaderModelOnly', 'title': 'BFS dedicated head swap — 4B only, strength 1', 'pos': [370, 1060], 'size': [330, 240], 'flags': {}, 'order': 24, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 22}], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [34]}], 'properties': {'Node name for S&R': 'LoraLoaderModelOnly'}, 'widgets_values': ['bfs_head_v1_flux-klein_4b.safetensors', 1.0]}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 7, 0, 9, 0, 'IMAGE'], [5, 9, 0, 10, 0, 'IMAGE'], [6, 4, 0, 10, 1, 'VAE'], [7, 5, 0, 11, 0, 'CONDITIONING'], [8, 10, 0, 11, 1, 'LATENT'], [9, 6, 0, 12, 0, 'CONDITIONING'], [10, 10, 0, 12, 1, 'LATENT'], [11, 8, 0, 13, 0, 'IMAGE'], [12, 13, 0, 14, 0, 'IMAGE'], [13, 4, 0, 14, 1, 'VAE'], [14, 11, 0, 15, 0, 'CONDITIONING'], [15, 14, 0, 15, 1, 'LATENT'], [16, 12, 0, 16, 0, 'CONDITIONING'], [17, 14, 0, 16, 1, 'LATENT'], [18, 1, 2, 17, 0, 'INT'], [19, 1, 3, 17, 1, 'INT'], [20, 1, 2, 18, 0, 'INT'], [21, 1, 3, 18, 1, 'INT'], [22, 2, 0, 25, 0, 'MODEL'], [23, 15, 0, 21, 1, 'CONDITIONING'], [24, 16, 0, 21, 2, 'CONDITIONING'], [25, 19, 0, 22, 0, 'NOISE'], [26, 21, 0, 22, 1, 'GUIDER'], [27, 20, 0, 22, 2, 'SAMPLER'], [28, 18, 0, 22, 3, 'SIGMAS'], [29, 17, 0, 22, 4, 'LATENT'], [30, 22, 0, 23, 0, 'LATENT'], [31, 4, 0, 23, 1, 'VAE'], [32, 23, 0, 24, 0, 'IMAGE'], [33, 1, 4, 24, 1, 'STRING'], [34, 25, 0, 21, 0, 'MODEL']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Approved Cast BFS Head Swap 4B': {'last_node_id': 25, 'last_link_id': 36, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [{'name': 'character_prefix', 'type': 'STRING', 'link': 4}], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [19, 21]}, {'name': 'height', 'type': 'INT', 'links': [20, 22]}, {'name': 'brief', 'type': 'STRING', 'links': [34]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', 'head_swap: Use Picture 1 as the base scene. Replace its head with the head from Picture 2, preserving the reference hair, eyes, nose and identity. Match the target head rotation, gaze and expression. Keep body, clothing, lighting, framing and background intact.', 'Episode default', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [23]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [7, 14, 32]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 8]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [5]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 1 — clean target / scene'}, {'id': 8, 'type': 'JJPLoadApprovedCast', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'reference_image', 'type': 'IMAGE', 'links': [12]}, {'name': 'character_prefix', 'type': 'STRING', 'links': [4]}, {'name': 'cast_record', 'type': 'STRING', 'links': [35]}], 'properties': {'Node name for S&R': 'JJPLoadApprovedCast'}, 'widgets_values': [], 'title': 'JJPLoadApprovedCast'}, {'id': 9, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 5}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [6]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 10, 'type': 'VAEEncode', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 6}, {'name': 'vae', 'type': 'VAE', 'link': 7}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [9, 11]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 11, 'type': 'ReferenceLatent', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 8}, {'name': 'latent', 'type': 'LATENT', 'link': 9}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [15]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 12, 'type': 'ReferenceLatent', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 10}, {'name': 'latent', 'type': 'LATENT', 'link': 11}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [17]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 13, 'type': 'ImageScaleToTotalPixels', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 12}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [13]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 14, 'type': 'VAEEncode', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 13}, {'name': 'vae', 'type': 'VAE', 'link': 14}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [16, 18]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 15, 'type': 'ReferenceLatent', 'pos': [1480, 660], 'size': [330, 280], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 15}, {'name': 'latent', 'type': 'LATENT', 'link': 16}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [24]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 16, 'type': 'ReferenceLatent', 'pos': [1850, 0], 'size': [330, 280], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 17}, {'name': 'latent', 'type': 'LATENT', 'link': 18}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [25]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 17, 'type': 'EmptyFlux2LatentImage', 'pos': [1850, 330], 'size': [330, 280], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 19}, {'name': 'height', 'type': 'INT', 'link': 20}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [30]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 18, 'type': 'Flux2Scheduler', 'pos': [1850, 660], 'size': [330, 280], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 21}, {'name': 'height', 'type': 'INT', 'link': 22}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [29]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 19, 'type': 'RandomNoise', 'pos': [2220, 0], 'size': [330, 280], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [26]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 20, 'type': 'KSamplerSelect', 'pos': [2220, 330], 'size': [330, 280], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [28]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 21, 'type': 'CFGGuider', 'pos': [2220, 660], 'size': [330, 280], 'flags': {}, 'order': 20, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 36}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 24}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 25}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [27]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 22, 'type': 'SamplerCustomAdvanced', 'pos': [2590, 0], 'size': [330, 280], 'flags': {}, 'order': 21, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 26}, {'name': 'guider', 'type': 'GUIDER', 'link': 27}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 28}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 29}, {'name': 'latent_image', 'type': 'LATENT', 'link': 30}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [31]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 23, 'type': 'VAEDecodeTiled', 'pos': [2590, 330], 'size': [330, 280], 'flags': {}, 'order': 22, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 31}, {'name': 'vae', 'type': 'VAE', 'link': 32}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [33]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 24, 'type': 'JJPArtworkExport', 'pos': [2590, 660], 'size': [330, 280], 'flags': {}, 'order': 23, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 33}, {'name': 'brief', 'type': 'STRING', 'link': 34}, {'name': 'cast_record', 'type': 'STRING', 'link': 35}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}, {'id': 25, 'type': 'LoraLoaderModelOnly', 'title': 'BFS dedicated head swap — 4B only, strength 1', 'pos': [370, 1060], 'size': [330, 240], 'flags': {}, 'order': 24, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 23}], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [36]}], 'properties': {'Node name for S&R': 'LoraLoaderModelOnly'}, 'widgets_values': ['bfs_head_v1_flux-klein_4b.safetensors', 1.0]}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 8, 1, 1, 0, 'STRING'], [5, 7, 0, 9, 0, 'IMAGE'], [6, 9, 0, 10, 0, 'IMAGE'], [7, 4, 0, 10, 1, 'VAE'], [8, 5, 0, 11, 0, 'CONDITIONING'], [9, 10, 0, 11, 1, 'LATENT'], [10, 6, 0, 12, 0, 'CONDITIONING'], [11, 10, 0, 12, 1, 'LATENT'], [12, 8, 0, 13, 0, 'IMAGE'], [13, 13, 0, 14, 0, 'IMAGE'], [14, 4, 0, 14, 1, 'VAE'], [15, 11, 0, 15, 0, 'CONDITIONING'], [16, 14, 0, 15, 1, 'LATENT'], [17, 12, 0, 16, 0, 'CONDITIONING'], [18, 14, 0, 16, 1, 'LATENT'], [19, 1, 2, 17, 0, 'INT'], [20, 1, 3, 17, 1, 'INT'], [21, 1, 2, 18, 0, 'INT'], [22, 1, 3, 18, 1, 'INT'], [23, 2, 0, 25, 0, 'MODEL'], [24, 15, 0, 21, 1, 'CONDITIONING'], [25, 16, 0, 21, 2, 'CONDITIONING'], [26, 19, 0, 22, 0, 'NOISE'], [27, 21, 0, 22, 1, 'GUIDER'], [28, 20, 0, 22, 2, 'SAMPLER'], [29, 18, 0, 22, 3, 'SIGMAS'], [30, 17, 0, 22, 4, 'LATENT'], [31, 22, 0, 23, 0, 'LATENT'], [32, 4, 0, 23, 1, 'VAE'], [33, 23, 0, 24, 0, 'IMAGE'], [34, 1, 4, 24, 1, 'STRING'], [35, 8, 2, 24, 2, 'STRING'], [36, 25, 0, 21, 0, 'MODEL']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Mesh Face Composite': {'last_node_id': 5, 'last_link_id': 4, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [1]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Source face — similar pose'}, {'id': 2, 'type': 'LoadImage', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Clean pre-swap target'}, {'id': 3, 'type': 'CommercialMeshSwap', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'source', 'type': 'IMAGE', 'link': 1}, {'name': 'target', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'image', 'type': 'IMAGE', 'links': [3]}, {'name': 'report', 'type': 'STRING', 'links': [4]}], 'properties': {'Node name for S&R': 'CommercialMeshSwap'}, 'widgets_values': [0, 0, 'Feather'], 'title': 'CommercialMeshSwap'}, {'id': 4, 'type': 'SaveImage', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 3}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudioCommercial/MeshComposite'], 'title': 'SaveImage'}, {'id': 5, 'type': 'PreviewAny', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'source', 'type': '*', 'link': 4}], 'outputs': [{'name': 'STRING', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'PreviewAny'}, 'widgets_values': [], 'title': 'PreviewAny'}], 'links': [[1, 1, 0, 3, 0, 'IMAGE'], [2, 2, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE'], [4, 3, 1, 5, 0, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Approved Cast Mesh Composite': {'last_node_id': 5, 'last_link_id': 4, 'nodes': [{'id': 1, 'type': 'JJPLoadApprovedCast', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'reference_image', 'type': 'IMAGE', 'links': [1]}, {'name': 'character_prefix', 'type': 'STRING', 'links': []}, {'name': 'cast_record', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPLoadApprovedCast'}, 'widgets_values': [], 'title': 'JJPLoadApprovedCast'}, {'id': 2, 'type': 'LoadImage', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Clean pre-swap target'}, {'id': 3, 'type': 'CommercialMeshSwap', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'source', 'type': 'IMAGE', 'link': 1}, {'name': 'target', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'image', 'type': 'IMAGE', 'links': [3]}, {'name': 'report', 'type': 'STRING', 'links': [4]}], 'properties': {'Node name for S&R': 'CommercialMeshSwap'}, 'widgets_values': [0, 0, 'Feather'], 'title': 'CommercialMeshSwap'}, {'id': 4, 'type': 'SaveImage', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 3}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudioCommercial/MeshComposite'], 'title': 'SaveImage'}, {'id': 5, 'type': 'PreviewAny', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'source', 'type': '*', 'link': 4}], 'outputs': [{'name': 'STRING', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'PreviewAny'}, 'widgets_values': [], 'title': 'PreviewAny'}], 'links': [[1, 1, 0, 3, 0, 'IMAGE'], [2, 2, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE'], [4, 3, 1, 5, 0, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Approved Cast Identity Edit': {'last_node_id': 24, 'last_link_id': 35, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [{'name': 'character_prefix', 'type': 'STRING', 'link': 4}], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [19, 21]}, {'name': 'height', 'type': 'INT', 'links': [20, 22]}, {'name': 'brief', 'type': 'STRING', 'links': [34]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', 'Use image 1 as the scene and composition. Replace only the selected person’s identity with the person from image 2. Preserve pose, clothing, lighting and the rest of the scene.', 'Episode default', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [23]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [7, 14, 32]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 8]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [5]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 1 — clean target / scene'}, {'id': 8, 'type': 'JJPLoadApprovedCast', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'reference_image', 'type': 'IMAGE', 'links': [12]}, {'name': 'character_prefix', 'type': 'STRING', 'links': [4]}, {'name': 'cast_record', 'type': 'STRING', 'links': [35]}], 'properties': {'Node name for S&R': 'JJPLoadApprovedCast'}, 'widgets_values': [], 'title': 'JJPLoadApprovedCast'}, {'id': 9, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 5}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [6]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 10, 'type': 'VAEEncode', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 6}, {'name': 'vae', 'type': 'VAE', 'link': 7}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [9, 11]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 11, 'type': 'ReferenceLatent', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 8}, {'name': 'latent', 'type': 'LATENT', 'link': 9}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [15]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 12, 'type': 'ReferenceLatent', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 10}, {'name': 'latent', 'type': 'LATENT', 'link': 11}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [17]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 13, 'type': 'ImageScaleToTotalPixels', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 12}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [13]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 14, 'type': 'VAEEncode', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 13}, {'name': 'vae', 'type': 'VAE', 'link': 14}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [16, 18]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 15, 'type': 'ReferenceLatent', 'pos': [1480, 660], 'size': [330, 280], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 15}, {'name': 'latent', 'type': 'LATENT', 'link': 16}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [24]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 16, 'type': 'ReferenceLatent', 'pos': [1850, 0], 'size': [330, 280], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 17}, {'name': 'latent', 'type': 'LATENT', 'link': 18}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [25]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 17, 'type': 'EmptyFlux2LatentImage', 'pos': [1850, 330], 'size': [330, 280], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 19}, {'name': 'height', 'type': 'INT', 'link': 20}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [30]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 18, 'type': 'Flux2Scheduler', 'pos': [1850, 660], 'size': [330, 280], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 21}, {'name': 'height', 'type': 'INT', 'link': 22}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [29]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 19, 'type': 'RandomNoise', 'pos': [2220, 0], 'size': [330, 280], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [26]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 20, 'type': 'KSamplerSelect', 'pos': [2220, 330], 'size': [330, 280], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [28]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 21, 'type': 'CFGGuider', 'pos': [2220, 660], 'size': [330, 280], 'flags': {}, 'order': 20, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 23}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 24}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 25}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [27]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 22, 'type': 'SamplerCustomAdvanced', 'pos': [2590, 0], 'size': [330, 280], 'flags': {}, 'order': 21, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 26}, {'name': 'guider', 'type': 'GUIDER', 'link': 27}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 28}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 29}, {'name': 'latent_image', 'type': 'LATENT', 'link': 30}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [31]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 23, 'type': 'VAEDecodeTiled', 'pos': [2590, 330], 'size': [330, 280], 'flags': {}, 'order': 22, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 31}, {'name': 'vae', 'type': 'VAE', 'link': 32}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [33]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 24, 'type': 'JJPArtworkExport', 'pos': [2590, 660], 'size': [330, 280], 'flags': {}, 'order': 23, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 33}, {'name': 'brief', 'type': 'STRING', 'link': 34}, {'name': 'cast_record', 'type': 'STRING', 'link': 35}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 8, 1, 1, 0, 'STRING'], [5, 7, 0, 9, 0, 'IMAGE'], [6, 9, 0, 10, 0, 'IMAGE'], [7, 4, 0, 10, 1, 'VAE'], [8, 5, 0, 11, 0, 'CONDITIONING'], [9, 10, 0, 11, 1, 'LATENT'], [10, 6, 0, 12, 0, 'CONDITIONING'], [11, 10, 0, 12, 1, 'LATENT'], [12, 8, 0, 13, 0, 'IMAGE'], [13, 13, 0, 14, 0, 'IMAGE'], [14, 4, 0, 14, 1, 'VAE'], [15, 11, 0, 15, 0, 'CONDITIONING'], [16, 14, 0, 15, 1, 'LATENT'], [17, 12, 0, 16, 0, 'CONDITIONING'], [18, 14, 0, 16, 1, 'LATENT'], [19, 1, 2, 17, 0, 'INT'], [20, 1, 3, 17, 1, 'INT'], [21, 1, 2, 18, 0, 'INT'], [22, 1, 3, 18, 1, 'INT'], [23, 2, 0, 21, 0, 'MODEL'], [24, 15, 0, 21, 1, 'CONDITIONING'], [25, 16, 0, 21, 2, 'CONDITIONING'], [26, 19, 0, 22, 0, 'NOISE'], [27, 21, 0, 22, 1, 'GUIDER'], [28, 20, 0, 22, 2, 'SAMPLER'], [29, 18, 0, 22, 3, 'SIGMAS'], [30, 17, 0, 22, 4, 'LATENT'], [31, 22, 0, 23, 0, 'LATENT'], [32, 4, 0, 23, 1, 'VAE'], [33, 23, 0, 24, 0, 'IMAGE'], [34, 1, 4, 24, 1, 'STRING'], [35, 8, 2, 24, 2, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Approved Duet Identity Edit': {'last_node_id': 30, 'last_link_id': 45, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [{'name': 'character_prefix', 'type': 'STRING', 'link': 7}], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [29, 31]}, {'name': 'height', 'type': 'INT', 'links': [30, 32]}, {'name': 'brief', 'type': 'STRING', 'links': [44]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', 'Use image 1 as the scene and composition. Use image 2 for Lead One and image 3 for Lead Two; keep their identities distinct.', 'Episode default', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [33]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [10, 17, 24, 42]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 11]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [13]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [8]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Image 1 — clean target / scene'}, {'id': 8, 'type': 'JJPLoadApprovedCast', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'reference_image', 'type': 'IMAGE', 'links': [15]}, {'name': 'character_prefix', 'type': 'STRING', 'links': []}, {'name': 'cast_record', 'type': 'STRING', 'links': [5]}], 'properties': {'Node name for S&R': 'JJPLoadApprovedCast'}, 'widgets_values': [], 'title': 'JJPLoadApprovedCast'}, {'id': 9, 'type': 'JJPLoadApprovedCast', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'reference_image', 'type': 'IMAGE', 'links': [22]}, {'name': 'character_prefix', 'type': 'STRING', 'links': []}, {'name': 'cast_record', 'type': 'STRING', 'links': [6]}], 'properties': {'Node name for S&R': 'JJPLoadApprovedCast'}, 'widgets_values': [], 'title': 'JJPLoadApprovedCast'}, {'id': 10, 'type': 'JJPCombineCast', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'lead_one', 'type': 'STRING', 'link': 5}, {'name': 'lead_two', 'type': 'STRING', 'link': 6}], 'outputs': [{'name': 'character_prefix', 'type': 'STRING', 'links': [7]}, {'name': 'cast_record', 'type': 'STRING', 'links': [45]}], 'properties': {'Node name for S&R': 'JJPCombineCast'}, 'widgets_values': [], 'title': 'JJPCombineCast'}, {'id': 11, 'type': 'ImageScaleToTotalPixels', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 8}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [9]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 12, 'type': 'VAEEncode', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 9}, {'name': 'vae', 'type': 'VAE', 'link': 10}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [12, 14]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 13, 'type': 'ReferenceLatent', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 11}, {'name': 'latent', 'type': 'LATENT', 'link': 12}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [18]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 14, 'type': 'ReferenceLatent', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 13}, {'name': 'latent', 'type': 'LATENT', 'link': 14}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [20]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 15, 'type': 'ImageScaleToTotalPixels', 'pos': [1480, 660], 'size': [330, 280], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 15}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [16]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 16, 'type': 'VAEEncode', 'pos': [1850, 0], 'size': [330, 280], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [19, 21]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 17, 'type': 'ReferenceLatent', 'pos': [1850, 330], 'size': [330, 280], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 18}, {'name': 'latent', 'type': 'LATENT', 'link': 19}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [25]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 18, 'type': 'ReferenceLatent', 'pos': [1850, 660], 'size': [330, 280], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 20}, {'name': 'latent', 'type': 'LATENT', 'link': 21}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [27]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 19, 'type': 'ImageScaleToTotalPixels', 'pos': [2220, 0], 'size': [330, 280], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 22}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [23]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 1], 'title': 'ImageScaleToTotalPixels'}, {'id': 20, 'type': 'VAEEncode', 'pos': [2220, 330], 'size': [330, 280], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 23}, {'name': 'vae', 'type': 'VAE', 'link': 24}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [26, 28]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 21, 'type': 'ReferenceLatent', 'pos': [2220, 660], 'size': [330, 280], 'flags': {}, 'order': 20, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 25}, {'name': 'latent', 'type': 'LATENT', 'link': 26}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [34]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 22, 'type': 'ReferenceLatent', 'pos': [2590, 0], 'size': [330, 280], 'flags': {}, 'order': 21, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 27}, {'name': 'latent', 'type': 'LATENT', 'link': 28}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [35]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 23, 'type': 'EmptyFlux2LatentImage', 'pos': [2590, 330], 'size': [330, 280], 'flags': {}, 'order': 22, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 29}, {'name': 'height', 'type': 'INT', 'link': 30}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [40]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 24, 'type': 'Flux2Scheduler', 'pos': [2590, 660], 'size': [330, 280], 'flags': {}, 'order': 23, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 31}, {'name': 'height', 'type': 'INT', 'link': 32}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [39]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 25, 'type': 'RandomNoise', 'pos': [2960, 0], 'size': [330, 280], 'flags': {}, 'order': 24, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [36]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 26, 'type': 'KSamplerSelect', 'pos': [2960, 330], 'size': [330, 280], 'flags': {}, 'order': 25, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [38]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 27, 'type': 'CFGGuider', 'pos': [2960, 660], 'size': [330, 280], 'flags': {}, 'order': 26, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 33}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 34}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 35}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [37]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 28, 'type': 'SamplerCustomAdvanced', 'pos': [3330, 0], 'size': [330, 280], 'flags': {}, 'order': 27, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 36}, {'name': 'guider', 'type': 'GUIDER', 'link': 37}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 38}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 39}, {'name': 'latent_image', 'type': 'LATENT', 'link': 40}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [41]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 29, 'type': 'VAEDecodeTiled', 'pos': [3330, 330], 'size': [330, 280], 'flags': {}, 'order': 28, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 41}, {'name': 'vae', 'type': 'VAE', 'link': 42}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [43]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 30, 'type': 'JJPArtworkExport', 'pos': [3330, 660], 'size': [330, 280], 'flags': {}, 'order': 29, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 43}, {'name': 'brief', 'type': 'STRING', 'link': 44}, {'name': 'cast_record', 'type': 'STRING', 'link': 45}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [5, 8, 2, 10, 0, 'STRING'], [6, 9, 2, 10, 1, 'STRING'], [7, 10, 0, 1, 0, 'STRING'], [8, 7, 0, 11, 0, 'IMAGE'], [9, 11, 0, 12, 0, 'IMAGE'], [10, 4, 0, 12, 1, 'VAE'], [11, 5, 0, 13, 0, 'CONDITIONING'], [12, 12, 0, 13, 1, 'LATENT'], [13, 6, 0, 14, 0, 'CONDITIONING'], [14, 12, 0, 14, 1, 'LATENT'], [15, 8, 0, 15, 0, 'IMAGE'], [16, 15, 0, 16, 0, 'IMAGE'], [17, 4, 0, 16, 1, 'VAE'], [18, 13, 0, 17, 0, 'CONDITIONING'], [19, 16, 0, 17, 1, 'LATENT'], [20, 14, 0, 18, 0, 'CONDITIONING'], [21, 16, 0, 18, 1, 'LATENT'], [22, 9, 0, 19, 0, 'IMAGE'], [23, 19, 0, 20, 0, 'IMAGE'], [24, 4, 0, 20, 1, 'VAE'], [25, 17, 0, 21, 0, 'CONDITIONING'], [26, 20, 0, 21, 1, 'LATENT'], [27, 18, 0, 22, 0, 'CONDITIONING'], [28, 20, 0, 22, 1, 'LATENT'], [29, 1, 2, 23, 0, 'INT'], [30, 1, 3, 23, 1, 'INT'], [31, 1, 2, 24, 0, 'INT'], [32, 1, 3, 24, 1, 'INT'], [33, 2, 0, 27, 0, 'MODEL'], [34, 21, 0, 27, 1, 'CONDITIONING'], [35, 22, 0, 27, 2, 'CONDITIONING'], [36, 25, 0, 28, 0, 'NOISE'], [37, 27, 0, 28, 1, 'GUIDER'], [38, 26, 0, 28, 2, 'SAMPLER'], [39, 24, 0, 28, 3, 'SIGMAS'], [40, 23, 0, 28, 4, 'LATENT'], [41, 28, 0, 29, 0, 'LATENT'], [42, 4, 0, 29, 1, 'VAE'], [43, 29, 0, 30, 0, 'IMAGE'], [44, 1, 4, 30, 1, 'STRING'], [45, 10, 1, 30, 2, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Upscale 2x': {'last_node_id': 5, 'last_link_id': 4, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 100], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload an image to enlarge'}, {'id': 2, 'type': 'UpscaleModelLoader', 'pos': [0, 440], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'UPSCALE_MODEL', 'type': 'UPSCALE_MODEL', 'links': [1]}], 'properties': {'Node name for S&R': 'UpscaleModelLoader'}, 'widgets_values': ['RealESRGAN_x4plus.pth']}, {'id': 3, 'type': 'ImageUpscaleWithModel', 'pos': [360, 100], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'upscale_model', 'type': 'UPSCALE_MODEL', 'link': 1}, {'name': 'image', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'ImageUpscaleWithModel'}, 'widgets_values': []}, {'id': 4, 'type': 'ImageScaleBy', 'pos': [720, 100], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}], 'properties': {'Node name for S&R': 'ImageScaleBy'}, 'widgets_values': ['lanczos', 0.5], 'title': '4x model then downsample to 2x'}, {'id': 5, 'type': 'SaveImage', 'pos': [1080, 100], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 4}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Upscale 2x']}], 'links': [[1, 2, 0, 3, 0, 'UPSCALE_MODEL'], [2, 1, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE'], [4, 4, 0, 5, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Upscale 4x': {'last_node_id': 4, 'last_link_id': 3, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 100], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload an image to enlarge'}, {'id': 2, 'type': 'UpscaleModelLoader', 'pos': [0, 440], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'UPSCALE_MODEL', 'type': 'UPSCALE_MODEL', 'links': [1]}], 'properties': {'Node name for S&R': 'UpscaleModelLoader'}, 'widgets_values': ['RealESRGAN_x4plus.pth']}, {'id': 3, 'type': 'ImageUpscaleWithModel', 'pos': [360, 100], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'upscale_model', 'type': 'UPSCALE_MODEL', 'link': 1}, {'name': 'image', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'ImageUpscaleWithModel'}, 'widgets_values': []}, {'id': 4, 'type': 'SaveImage', 'pos': [1080, 100], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 3}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Upscale 4x']}], 'links': [[1, 2, 0, 3, 0, 'UPSCALE_MODEL'], [2, 1, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Restore SeedVR2': {'last_node_id': 11, 'last_link_id': 15, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 100], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [1]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload an image to enlarge'}, {'id': 2, 'type': 'ImageScaleBy', 'pos': [360, 100], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 1}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2, 14]}], 'properties': {'Node name for S&R': 'ImageScaleBy'}, 'widgets_values': ['lanczos', 2.0], 'title': '2x target; start with source at 1024px or smaller'}, {'id': 3, 'type': 'SeedVR2Preprocess', 'pos': [720, 100], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'resized_images', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'images', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'SeedVR2Preprocess'}, 'widgets_values': []}, {'id': 4, 'type': 'VAELoader', 'pos': [360, 440], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [4, 12]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['seedvr2_ema_vae_fp16.safetensors']}, {'id': 5, 'type': 'UNETLoader', 'pos': [0, 440], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [5, 7]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['seedvr2_3b_int8_convrot.safetensors', 'default']}, {'id': 6, 'type': 'VAEEncodeTiled', 'pos': [1080, 100], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 3}, {'name': 'vae', 'type': 'VAE', 'link': 4}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [6, 10]}], 'properties': {'Node name for S&R': 'VAEEncodeTiled'}, 'widgets_values': [512, 128, 64, 8]}, {'id': 7, 'type': 'SeedVR2Conditioning', 'pos': [1080, 440], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 5}, {'name': 'vae_conditioning', 'type': 'LATENT', 'link': 6}], 'outputs': [{'name': 'positive', 'type': 'CONDITIONING', 'links': [8]}, {'name': 'negative', 'type': 'CONDITIONING', 'links': [9]}], 'properties': {'Node name for S&R': 'SeedVR2Conditioning'}, 'widgets_values': []}, {'id': 8, 'type': 'KSampler', 'pos': [1440, 100], 'size': [320, 250], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 7}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 8}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 9}, {'name': 'latent_image', 'type': 'LATENT', 'link': 10}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [11]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'fixed', 1, 1.0, 'euler', 'simple', 1.0]}, {'id': 9, 'type': 'VAEDecodeTiled', 'pos': [1800, 100], 'size': [320, 250], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 11}, {'name': 'vae', 'type': 'VAE', 'link': 12}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [13]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 128, 64, 8]}, {'id': 10, 'type': 'SeedVR2PostProcessing', 'pos': [2160, 100], 'size': [320, 250], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 13}, {'name': 'original_resized_images', 'type': 'IMAGE', 'link': 14}], 'outputs': [{'name': 'images', 'type': 'IMAGE', 'links': [15]}], 'properties': {'Node name for S&R': 'SeedVR2PostProcessing'}, 'widgets_values': ['none']}, {'id': 11, 'type': 'SaveImage', 'pos': [2520, 100], 'size': [320, 250], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 15}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Restore SeedVR2']}], 'links': [[1, 1, 0, 2, 0, 'IMAGE'], [2, 2, 0, 3, 0, 'IMAGE'], [3, 3, 0, 6, 0, 'IMAGE'], [4, 4, 0, 6, 1, 'VAE'], [5, 5, 0, 7, 0, 'MODEL'], [6, 6, 0, 7, 1, 'LATENT'], [7, 5, 0, 8, 0, 'MODEL'], [8, 7, 0, 8, 1, 'CONDITIONING'], [9, 7, 1, 8, 2, 'CONDITIONING'], [10, 6, 0, 8, 3, 'LATENT'], [11, 8, 0, 9, 0, 'LATENT'], [12, 4, 0, 9, 1, 'VAE'], [13, 9, 0, 10, 0, 'IMAGE'], [14, 2, 0, 10, 1, 'IMAGE'], [15, 10, 0, 11, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Approve Cast': {'last_node_id': 2, 'last_link_id': 1, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [1]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'LoadImage'}, {'id': 2, 'type': 'JJPSaveApprovedCast', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 1}], 'outputs': [{'name': 'reference', 'type': 'IMAGE', 'links': []}, {'name': 'record', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPSaveApprovedCast'}, 'widgets_values': ['Lead One', 'Approved reference', '', False], 'title': 'JJPSaveApprovedCast'}], 'links': [[1, 1, 0, 2, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Delivery Export': {'last_node_id': 3, 'last_link_id': 2, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [1]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'LoadImage'}, {'id': 2, 'type': 'JJPEpisodeBrief', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': []}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': []}, {'name': 'height', 'type': 'INT', 'links': []}, {'name': 'brief', 'type': 'STRING', 'links': [2]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Custom', 'Square cover', '', 'None', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 3, 'type': 'JJPArtworkExport', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 1}, {'name': 'brief', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 1, 0, 3, 0, 'IMAGE'], [2, 2, 4, 3, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'JJP Episode - Verdict of the Lamb': {'last_node_id': 14, 'last_link_id': 19, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [4, 6]}, {'name': 'height', 'type': 'INT', 'links': [5, 7]}, {'name': 'brief', 'type': 'STRING', 'links': [19]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Verdict of the Lamb', 'Square cover', 'A cinematic music video still.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 8, 'type': 'Flux2Scheduler', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 6}, {'name': 'height', 'type': 'INT', 'link': 7}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [14]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 9, 'type': 'RandomNoise', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [11]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 10, 'type': 'KSamplerSelect', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [13]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 11, 'type': 'CFGGuider', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [12]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 12, 'type': 'SamplerCustomAdvanced', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 11}, {'name': 'guider', 'type': 'GUIDER', 'link': 12}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 13}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 14}, {'name': 'latent_image', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [16]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 13, 'type': 'VAEDecodeTiled', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 14, 'type': 'JJPArtworkExport', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}, {'name': 'brief', 'type': 'STRING', 'link': 19}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 1, 2, 7, 0, 'INT'], [5, 1, 3, 7, 1, 'INT'], [6, 1, 2, 8, 0, 'INT'], [7, 1, 3, 8, 1, 'INT'], [8, 2, 0, 11, 0, 'MODEL'], [9, 5, 0, 11, 1, 'CONDITIONING'], [10, 6, 0, 11, 2, 'CONDITIONING'], [11, 9, 0, 12, 0, 'NOISE'], [12, 11, 0, 12, 1, 'GUIDER'], [13, 10, 0, 12, 2, 'SAMPLER'], [14, 8, 0, 12, 3, 'SIGMAS'], [15, 7, 0, 12, 4, 'LATENT'], [16, 12, 0, 13, 0, 'LATENT'], [17, 4, 0, 13, 1, 'VAE'], [18, 13, 0, 14, 0, 'IMAGE'], [19, 1, 4, 14, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'JJP Episode - Anastasis': {'last_node_id': 14, 'last_link_id': 19, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [4, 6]}, {'name': 'height', 'type': 'INT', 'links': [5, 7]}, {'name': 'brief', 'type': 'STRING', 'links': [19]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Anastasis', 'Square cover', 'A cinematic music video still.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 8, 'type': 'Flux2Scheduler', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 6}, {'name': 'height', 'type': 'INT', 'link': 7}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [14]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 9, 'type': 'RandomNoise', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [11]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 10, 'type': 'KSamplerSelect', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [13]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 11, 'type': 'CFGGuider', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [12]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 12, 'type': 'SamplerCustomAdvanced', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 11}, {'name': 'guider', 'type': 'GUIDER', 'link': 12}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 13}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 14}, {'name': 'latent_image', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [16]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 13, 'type': 'VAEDecodeTiled', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 14, 'type': 'JJPArtworkExport', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}, {'name': 'brief', 'type': 'STRING', 'link': 19}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 1, 2, 7, 0, 'INT'], [5, 1, 3, 7, 1, 'INT'], [6, 1, 2, 8, 0, 'INT'], [7, 1, 3, 8, 1, 'INT'], [8, 2, 0, 11, 0, 'MODEL'], [9, 5, 0, 11, 1, 'CONDITIONING'], [10, 6, 0, 11, 2, 'CONDITIONING'], [11, 9, 0, 12, 0, 'NOISE'], [12, 11, 0, 12, 1, 'GUIDER'], [13, 10, 0, 12, 2, 'SAMPLER'], [14, 8, 0, 12, 3, 'SIGMAS'], [15, 7, 0, 12, 4, 'LATENT'], [16, 12, 0, 13, 0, 'LATENT'], [17, 4, 0, 13, 1, 'VAE'], [18, 13, 0, 14, 0, 'IMAGE'], [19, 1, 4, 14, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'JJP Episode - God-Breathed': {'last_node_id': 14, 'last_link_id': 19, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [4, 6]}, {'name': 'height', 'type': 'INT', 'links': [5, 7]}, {'name': 'brief', 'type': 'STRING', 'links': [19]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['God-Breathed', 'Square cover', 'A cinematic music video still.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 8, 'type': 'Flux2Scheduler', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 6}, {'name': 'height', 'type': 'INT', 'link': 7}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [14]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 9, 'type': 'RandomNoise', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [11]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 10, 'type': 'KSamplerSelect', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [13]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 11, 'type': 'CFGGuider', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [12]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 12, 'type': 'SamplerCustomAdvanced', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 11}, {'name': 'guider', 'type': 'GUIDER', 'link': 12}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 13}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 14}, {'name': 'latent_image', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [16]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 13, 'type': 'VAEDecodeTiled', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 14, 'type': 'JJPArtworkExport', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}, {'name': 'brief', 'type': 'STRING', 'link': 19}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 1, 2, 7, 0, 'INT'], [5, 1, 3, 7, 1, 'INT'], [6, 1, 2, 8, 0, 'INT'], [7, 1, 3, 8, 1, 'INT'], [8, 2, 0, 11, 0, 'MODEL'], [9, 5, 0, 11, 1, 'CONDITIONING'], [10, 6, 0, 11, 2, 'CONDITIONING'], [11, 9, 0, 12, 0, 'NOISE'], [12, 11, 0, 12, 1, 'GUIDER'], [13, 10, 0, 12, 2, 'SAMPLER'], [14, 8, 0, 12, 3, 'SIGMAS'], [15, 7, 0, 12, 4, 'LATENT'], [16, 12, 0, 13, 0, 'LATENT'], [17, 4, 0, 13, 1, 'VAE'], [18, 13, 0, 14, 0, 'IMAGE'], [19, 1, 4, 14, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'JJP Episode - Every Thought Captive': {'last_node_id': 14, 'last_link_id': 19, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [4, 6]}, {'name': 'height', 'type': 'INT', 'links': [5, 7]}, {'name': 'brief', 'type': 'STRING', 'links': [19]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Every Thought Captive', 'Square cover', 'A cinematic music video still.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 8, 'type': 'Flux2Scheduler', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 6}, {'name': 'height', 'type': 'INT', 'link': 7}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [14]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 9, 'type': 'RandomNoise', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [11]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 10, 'type': 'KSamplerSelect', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [13]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 11, 'type': 'CFGGuider', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [12]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 12, 'type': 'SamplerCustomAdvanced', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 11}, {'name': 'guider', 'type': 'GUIDER', 'link': 12}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 13}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 14}, {'name': 'latent_image', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [16]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 13, 'type': 'VAEDecodeTiled', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 14, 'type': 'JJPArtworkExport', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}, {'name': 'brief', 'type': 'STRING', 'link': 19}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 1, 2, 7, 0, 'INT'], [5, 1, 3, 7, 1, 'INT'], [6, 1, 2, 8, 0, 'INT'], [7, 1, 3, 8, 1, 'INT'], [8, 2, 0, 11, 0, 'MODEL'], [9, 5, 0, 11, 1, 'CONDITIONING'], [10, 6, 0, 11, 2, 'CONDITIONING'], [11, 9, 0, 12, 0, 'NOISE'], [12, 11, 0, 12, 1, 'GUIDER'], [13, 10, 0, 12, 2, 'SAMPLER'], [14, 8, 0, 12, 3, 'SIGMAS'], [15, 7, 0, 12, 4, 'LATENT'], [16, 12, 0, 13, 0, 'LATENT'], [17, 4, 0, 13, 1, 'VAE'], [18, 13, 0, 14, 0, 'IMAGE'], [19, 1, 4, 14, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'JJP Episode - Present Your Body': {'last_node_id': 14, 'last_link_id': 19, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [4, 6]}, {'name': 'height', 'type': 'INT', 'links': [5, 7]}, {'name': 'brief', 'type': 'STRING', 'links': [19]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Present Your Body', 'Square cover', 'A cinematic music video still.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 8, 'type': 'Flux2Scheduler', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 6}, {'name': 'height', 'type': 'INT', 'link': 7}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [14]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 9, 'type': 'RandomNoise', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [11]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 10, 'type': 'KSamplerSelect', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [13]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 11, 'type': 'CFGGuider', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [12]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 12, 'type': 'SamplerCustomAdvanced', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 11}, {'name': 'guider', 'type': 'GUIDER', 'link': 12}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 13}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 14}, {'name': 'latent_image', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [16]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 13, 'type': 'VAEDecodeTiled', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 14, 'type': 'JJPArtworkExport', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}, {'name': 'brief', 'type': 'STRING', 'link': 19}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 1, 2, 7, 0, 'INT'], [5, 1, 3, 7, 1, 'INT'], [6, 1, 2, 8, 0, 'INT'], [7, 1, 3, 8, 1, 'INT'], [8, 2, 0, 11, 0, 'MODEL'], [9, 5, 0, 11, 1, 'CONDITIONING'], [10, 6, 0, 11, 2, 'CONDITIONING'], [11, 9, 0, 12, 0, 'NOISE'], [12, 11, 0, 12, 1, 'GUIDER'], [13, 10, 0, 12, 2, 'SAMPLER'], [14, 8, 0, 12, 3, 'SIGMAS'], [15, 7, 0, 12, 4, 'LATENT'], [16, 12, 0, 13, 0, 'LATENT'], [17, 4, 0, 13, 1, 'VAE'], [18, 13, 0, 14, 0, 'IMAGE'], [19, 1, 4, 14, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'JJP Episode - Jesus, Thank You For Loving Me': {'last_node_id': 14, 'last_link_id': 19, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [4, 6]}, {'name': 'height', 'type': 'INT', 'links': [5, 7]}, {'name': 'brief', 'type': 'STRING', 'links': [19]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Jesus, Thank You For Loving Me', 'Square cover', 'A cinematic music video still.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 8, 'type': 'Flux2Scheduler', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 6}, {'name': 'height', 'type': 'INT', 'link': 7}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [14]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 9, 'type': 'RandomNoise', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [11]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 10, 'type': 'KSamplerSelect', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [13]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 11, 'type': 'CFGGuider', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [12]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 12, 'type': 'SamplerCustomAdvanced', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 11}, {'name': 'guider', 'type': 'GUIDER', 'link': 12}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 13}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 14}, {'name': 'latent_image', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [16]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 13, 'type': 'VAEDecodeTiled', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 14, 'type': 'JJPArtworkExport', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}, {'name': 'brief', 'type': 'STRING', 'link': 19}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 1, 2, 7, 0, 'INT'], [5, 1, 3, 7, 1, 'INT'], [6, 1, 2, 8, 0, 'INT'], [7, 1, 3, 8, 1, 'INT'], [8, 2, 0, 11, 0, 'MODEL'], [9, 5, 0, 11, 1, 'CONDITIONING'], [10, 6, 0, 11, 2, 'CONDITIONING'], [11, 9, 0, 12, 0, 'NOISE'], [12, 11, 0, 12, 1, 'GUIDER'], [13, 10, 0, 12, 2, 'SAMPLER'], [14, 8, 0, 12, 3, 'SIGMAS'], [15, 7, 0, 12, 4, 'LATENT'], [16, 12, 0, 13, 0, 'LATENT'], [17, 4, 0, 13, 1, 'VAE'], [18, 13, 0, 14, 0, 'IMAGE'], [19, 1, 4, 14, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'JJP Episode - Mirror of Glory': {'last_node_id': 14, 'last_link_id': 19, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [4, 6]}, {'name': 'height', 'type': 'INT', 'links': [5, 7]}, {'name': 'brief', 'type': 'STRING', 'links': [19]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Mirror of Glory', 'Square cover', 'A cinematic music video still.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 8, 'type': 'Flux2Scheduler', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 6}, {'name': 'height', 'type': 'INT', 'link': 7}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [14]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 9, 'type': 'RandomNoise', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [11]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 10, 'type': 'KSamplerSelect', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [13]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 11, 'type': 'CFGGuider', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [12]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 12, 'type': 'SamplerCustomAdvanced', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 11}, {'name': 'guider', 'type': 'GUIDER', 'link': 12}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 13}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 14}, {'name': 'latent_image', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [16]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 13, 'type': 'VAEDecodeTiled', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 14, 'type': 'JJPArtworkExport', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}, {'name': 'brief', 'type': 'STRING', 'link': 19}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 1, 2, 7, 0, 'INT'], [5, 1, 3, 7, 1, 'INT'], [6, 1, 2, 8, 0, 'INT'], [7, 1, 3, 8, 1, 'INT'], [8, 2, 0, 11, 0, 'MODEL'], [9, 5, 0, 11, 1, 'CONDITIONING'], [10, 6, 0, 11, 2, 'CONDITIONING'], [11, 9, 0, 12, 0, 'NOISE'], [12, 11, 0, 12, 1, 'GUIDER'], [13, 10, 0, 12, 2, 'SAMPLER'], [14, 8, 0, 12, 3, 'SIGMAS'], [15, 7, 0, 12, 4, 'LATENT'], [16, 12, 0, 13, 0, 'LATENT'], [17, 4, 0, 13, 1, 'VAE'], [18, 13, 0, 14, 0, 'IMAGE'], [19, 1, 4, 14, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'JJP Episode - Love Your Enemies': {'last_node_id': 14, 'last_link_id': 19, 'nodes': [{'id': 1, 'type': 'JJPEpisodeBrief', 'pos': [0, 0], 'size': [330, 280], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'positive', 'type': 'STRING', 'links': [2]}, {'name': 'negative', 'type': 'STRING', 'links': []}, {'name': 'width', 'type': 'INT', 'links': [4, 6]}, {'name': 'height', 'type': 'INT', 'links': [5, 7]}, {'name': 'brief', 'type': 'STRING', 'links': [19]}], 'properties': {'Node name for S&R': 'JJPEpisodeBrief'}, 'widgets_values': ['Love Your Enemies', 'Square cover', 'A cinematic music video still.', 'Episode default', '', '', '', True, '', '', ''], 'title': 'JJPEpisodeBrief'}, {'id': 2, 'type': 'UNETLoader', 'pos': [0, 330], 'size': [330, 280], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [8]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 3, 'type': 'CLIPLoader', 'pos': [0, 660], 'size': [330, 280], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 4, 'type': 'VAELoader', 'pos': [370, 0], 'size': [330, 280], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 5, 'type': 'CLIPTextEncode', 'pos': [370, 330], 'size': [330, 280], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}, {'name': 'text', 'type': 'STRING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [3, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': [], 'title': 'CLIPTextEncode'}, {'id': 6, 'type': 'ConditioningZeroOut', 'pos': [370, 660], 'size': [330, 280], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 3}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [10]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 7, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 0], 'size': [330, 280], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [15]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 8, 'type': 'Flux2Scheduler', 'pos': [740, 330], 'size': [330, 280], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 6}, {'name': 'height', 'type': 'INT', 'link': 7}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [14]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 9, 'type': 'RandomNoise', 'pos': [740, 660], 'size': [330, 280], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [11]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'RandomNoise'}, {'id': 10, 'type': 'KSamplerSelect', 'pos': [1110, 0], 'size': [330, 280], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [13]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 11, 'type': 'CFGGuider', 'pos': [1110, 330], 'size': [330, 280], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [12]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 12, 'type': 'SamplerCustomAdvanced', 'pos': [1110, 660], 'size': [330, 280], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 11}, {'name': 'guider', 'type': 'GUIDER', 'link': 12}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 13}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 14}, {'name': 'latent_image', 'type': 'LATENT', 'link': 15}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [16]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 13, 'type': 'VAEDecodeTiled', 'pos': [1480, 0], 'size': [330, 280], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 16}, {'name': 'vae', 'type': 'VAE', 'link': 17}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [18]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 14, 'type': 'JJPArtworkExport', 'pos': [1480, 330], 'size': [330, 280], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 18}, {'name': 'brief', 'type': 'STRING', 'link': 19}], 'outputs': [{'name': 'delivery', 'type': 'IMAGE', 'links': []}, {'name': 'saved_files', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'JJPArtworkExport'}, 'widgets_values': ['From brief', True, 2048], 'title': 'JJPArtworkExport'}], 'links': [[1, 3, 0, 5, 0, 'CLIP'], [2, 1, 0, 5, 1, 'STRING'], [3, 5, 0, 6, 0, 'CONDITIONING'], [4, 1, 2, 7, 0, 'INT'], [5, 1, 3, 7, 1, 'INT'], [6, 1, 2, 8, 0, 'INT'], [7, 1, 3, 8, 1, 'INT'], [8, 2, 0, 11, 0, 'MODEL'], [9, 5, 0, 11, 1, 'CONDITIONING'], [10, 6, 0, 11, 2, 'CONDITIONING'], [11, 9, 0, 12, 0, 'NOISE'], [12, 11, 0, 12, 1, 'GUIDER'], [13, 10, 0, 12, 2, 'SAMPLER'], [14, 8, 0, 12, 3, 'SIGMAS'], [15, 7, 0, 12, 4, 'LATENT'], [16, 12, 0, 13, 0, 'LATENT'], [17, 4, 0, 13, 1, 'VAE'], [18, 13, 0, 14, 0, 'IMAGE'], [19, 1, 4, 14, 1, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.5, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}}
COMFY_REV='1332fabfcc507aa7c548cd5152071ed494c207e8'
CF_SHA='1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872'
# Isolate reviewed code/models from historical research-only installations.
BASE=Path('/kaggle/temp/image-studio-commercial-v7'); BASE.mkdir(parents=True,exist_ok=True)
OUT=Path('/kaggle/working/image-studio-commercial-v7'); OUT.mkdir(parents=True,exist_ok=True)
lock=open(BASE/'startup.lock','w')
try: fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
except BlockingIOError: raise RuntimeError('Image Studio is already running. Stop its startup cell before starting again.')
children=[]
# Fail before downloading if the previous studio still owns this port.
import socket
probe=socket.socket()
try: probe.bind(('127.0.0.1',8188))
except OSError: raise RuntimeError('Port 8188 is occupied. Stop the earlier studio cell before starting commercial v7.')
finally: probe.close()

def run(args,**kw):subprocess.run(list(map(str,args)),check=True,**kw)
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()
def download(url,path,expected,size=None):
 if path.exists() and sha(path)==expected:return
 if path.exists():raise RuntimeError(f'Checksum mismatch: {path}. Preserve or move this file before retrying.')
 if size and shutil.disk_usage(path.parent).free<size+2*1024**3:raise RuntimeError('Insufficient free disk for model plus 2 GiB reserve.')
 temp=path.with_suffix(path.suffix+'.partial')
 print('Downloading',path.name,flush=True)
 with urllib.request.urlopen(url,timeout=120) as response,open(temp,'wb') as target:shutil.copyfileobj(response,target,8*1024*1024)
 if sha(temp)!=expected:raise RuntimeError(f'Download checksum mismatch: {temp}')
 temp.replace(path)
def spawn(args,log):
 f=open(log,'w');p=subprocess.Popen(list(map(str,args)),stdout=f,stderr=subprocess.STDOUT,start_new_session=True);f.close();children.append(p);return p
# Only allowlisted weights can enter this fresh installation. Never reuse old plugins.
assets=[a for a in CATALOG['assets'] if (a['pack']!='create_edit' or ENABLE_CREATE_EDIT) and (not a['name'].startswith('seedvr2') or ENABLE_SEEDVR2) and (not a['name'].endswith('.task') or ENABLE_MESH_SWAP) and (a['folder']!='loras' or (ENABLE_CREATE_EDIT and ENABLE_BFS_HEADSWAP))]
resolved_models={};missing=[]
for asset in assets:
 name=asset['name'];candidates=[]
 for root in [Path('/kaggle/input'),Path('/kaggle/working/image-studio-model-packs'),BASE/'ComfyUI/models']:
  if root.exists():candidates.extend(root.rglob(name))
 valid=next((p for p in candidates if p.is_file() and p.stat().st_size==asset['bytes'] and sha(p)==asset['sha256']),None)
 if valid:resolved_models[name]=valid
 else:missing.append(name)
if missing and REQUIRE_ATTACHED_MODELS:
 fcntl.flock(lock,fcntl.LOCK_UN);lock.close()
 raise RuntimeError('Run Commercial Model Packs on CPU for create_edit and finish; attach the saved private datasets or notebook outputs. Missing: '+', '.join(missing)+'. No model downloads started.')
try:
 import torch
 if not torch.cuda.is_available():raise RuntimeError('Select a Kaggle GPU accelerator and restart this cell.')
 print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,flush=True)
 free,total=torch.cuda.mem_get_info(0)
 if free<10*1024**3:raise RuntimeError('GPU 0 has less than 10 GiB free. Stop other GPU notebooks/processes first.')
 # Separate T4s do not form one shared VRAM pool. This studio uses GPU 0.
 comfy=BASE/'ComfyUI'
 if not comfy.exists():
  run(['git','clone','--no-checkout','https://github.com/comfyanonymous/ComfyUI.git',comfy])
  run(['git','-C',comfy,'checkout','--detach',COMFY_REV])
 if subprocess.check_output(['git','-C',str(comfy),'rev-parse','HEAD'],text=True).strip()!=COMFY_REV:raise RuntimeError('Unexpected ComfyUI revision.')
 venv=BASE/'venv';python=venv/'bin/python'
 if not python.exists():run([sys.executable,'-m','venv','--system-site-packages','--without-pip',venv])
 constraints=BASE/'constraints.txt'
 import importlib.metadata as meta
 protected=['torch','torchvision','torchaudio','numpy','scipy']
 constraints.write_text('\n'.join(f'{p}=={meta.version(p)}' for p in protected))
 requirements=BASE/'requirements.txt'
 requirements.write_text('\n'.join(x for x in (comfy/'requirements.txt').read_text().splitlines() if x.strip() not in protected)+'\ntransformers==5.22.2\nhuggingface-hub==2.4.0\n')
 fingerprint=hashlib.sha256((requirements.read_text()+constraints.read_text()+sys.version).encode()).hexdigest()
 marker=BASE/'setup.complete'
 if not marker.exists() or marker.read_text()!=fingerprint:
  run([sys.executable,'-m','pip','--python',python,'install','-r',requirements,'-c',constraints])
  run([python,'-c','import torch, torchvision, transformers, aiohttp, safetensors; assert torch.cuda.is_available()'])
  marker.write_text(fingerprint)
 if ENABLE_MESH_SWAP:
  # Some minimal Linux images lack GLES, required even for MediaPipe CPU inference.
  import ctypes.util
  if not ctypes.util.find_library('GLESv2'):
   deb=BASE/'libgles2.deb'
   download('https://deb.debian.org/debian/pool/main/libg/libglvnd/libgles2_1.7.0-1+b2_amd64.deb',deb,'e12c18aa77939d122ee1dea520ba0440b97dc19200b7396e03dff03a3ece74c1')
   run(['dpkg-deb','-x',deb,BASE/'gles'])
   os.environ['LD_LIBRARY_PATH']=str(BASE/'gles/usr/lib/x86_64-linux-gnu')+':'+os.environ.get('LD_LIBRARY_PATH','')
  face_marker=BASE/'mediapipe-setup.complete'
  if not face_marker.exists() or face_marker.read_text()!=fingerprint:
   run([sys.executable,'-m','pip','--python',python,'install','mediapipe==1.2.2','-c',constraints])
   face_marker.write_text(fingerprint)
  run([python,'-c','import mediapipe as mp; mp.tasks.vision.FaceLandmarkerOptions'])
 custom=comfy/'custom_nodes/commercial_image_tools';custom.mkdir(parents=True,exist_ok=True)
 (custom/'__init__.py').write_text(FACE_NODE_SOURCE)
 allowed={'commercial_image_tools','jjp_image_production','__pycache__'}
 unexpected=[p.name for p in (comfy/'custom_nodes').iterdir() if p.is_dir() and p.name not in allowed and not p.name.startswith('.')]
 if unexpected:raise RuntimeError('Unexpected custom node directories in isolated commercial installation: '+', '.join(unexpected))
 custom_jjp=comfy/'custom_nodes/jjp_image_production';custom_jjp.mkdir(parents=True,exist_ok=True)
 (custom_jjp/'__init__.py').write_text(JJP_NODE_SOURCE)
 (custom_jjp/'jjp_presets.json').write_text(JJP_PRESETS_SOURCE)
 run([sys.executable,'-m','pip','--python',python,'freeze'],stdout=open(OUT/'packages.txt','w'))
 for asset in assets:
  dest=comfy/'models'/asset['folder']/asset['name'];dest.parent.mkdir(parents=True,exist_ok=True)
  if dest.is_symlink() and not dest.exists():dest.unlink()
  if not dest.exists() and asset['name'] in resolved_models:dest.symlink_to(resolved_models[asset['name']])
  if not dest.exists():download(asset['source'],dest,asset['sha256'],asset['bytes'])
  if sha(dest)!=asset['sha256']:raise RuntimeError('Asset checksum mismatch: '+asset['name'])
 def workflow_enabled(wf):
  types={n['type'] for n in wf['nodes']}
  return (ENABLE_CREATE_EDIT or 'EmptyFlux2LatentImage' not in types) and (ENABLE_MESH_SWAP or 'CommercialMeshSwap' not in types) and (ENABLE_SEEDVR2 or 'SeedVR2Conditioning' not in types) and (ENABLE_BFS_HEADSWAP or 'LoraLoaderModelOnly' not in types)
 WORKFLOW_PAYLOAD={k:v for k,v in WORKFLOW_PAYLOAD.items() if workflow_enabled(v)}
 workflow_dir=comfy/'user/default/workflows';workflow_dir.mkdir(parents=True,exist_ok=True)
 for name,wf in WORKFLOW_PAYLOAD.items():(workflow_dir/f'Image Studio - {name}.json').write_text(json.dumps(wf,indent=2))
 # Stable output destination: PNGs retain the actual graph, prompt, seed and settings.
 for name,content in LICENSE_FILES.items():(OUT/name).write_text(content)
 (OUT/'provenance.json').write_text(json.dumps({'profile':'commercial-v7','comfy_revision':COMFY_REV,'assets':assets,'torch':torch.__version__},indent=2))
 cf=BASE/'cloudflared'
 download('https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64',cf,CF_SHA)
 cf.chmod(0o700)
 server=spawn([python,comfy/'main.py','--listen','127.0.0.1','--port','8188','--lowvram','--force-fp16','--fp16-text-enc','--preview-method','none','--output-directory',OUT/'images','--disable-auto-launch','--disable-all-custom-nodes','--whitelist-custom-nodes','commercial_image_tools','jjp_image_production'],OUT/'comfy.log')
 deadline=time.monotonic()+180
 while True:
  if server.poll() is not None:raise RuntimeError('ComfyUI stopped. See comfy.log in Kaggle outputs.')
  try:
   with urllib.request.urlopen('http://127.0.0.1:8188/object_info',timeout=3) as r:info=json.load(r)
   break
  except OSError:
   if time.monotonic()>deadline:raise RuntimeError('ComfyUI startup timed out. See comfy.log.')
   time.sleep(1)
 # Validate installed node classes before offering the browser link.
 for wf in WORKFLOW_PAYLOAD.values():
  for node in wf['nodes']:
   if node['type'] not in info:raise RuntimeError('Required node unavailable: '+node['type'])
 tunnel=spawn([cf,'tunnel','--url','http://127.0.0.1:8188','--no-autoupdate'],OUT/'tunnel.log')
 deadline=time.monotonic()+90
 url=None
 while time.monotonic()<deadline:
  text=(OUT/'tunnel.log').read_text()
  matches=re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
  if matches:url=matches[-1];break
  if tunnel.poll() is not None:raise RuntimeError('Cloudflare stopped; inspect tunnel.log.')
  time.sleep(1)
 if not url:raise RuntimeError('No Cloudflare URL received; inspect tunnel.log.')
 print('\nOPEN IMAGE STUDIO:',url,flush=True)
 print('Workflow menu → Image Studio: '+', '.join(WORKFLOW_PAYLOAD)+'. Select a workflow, then press Run.')
 print('No generation has started. Keep this cell running. Anyone with this session URL can access the studio.')
 print('Save Version with outputs before ending Kaggle. Images:',OUT/'images',flush=True)
 while server.poll() is None and tunnel.poll() is None:time.sleep(2)
 raise RuntimeError('Studio or tunnel stopped; see the saved logs.')
finally:
 for p in reversed(children):
  if p.poll() is None:
   os.killpg(p.pid,signal.SIGTERM)
   try:p.wait(timeout=10)
   except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait()
 fcntl.flock(lock,fcntl.LOCK_UN);lock.close()
